# LeetCode 317 — Shortest Distance from All Buildings bằng Python (Hard, Premium)

> **Lưu ý:** đây là bài **Premium** (chỉ tài khoản LeetCode Premium mới mở được). Phần đề, ví dụ, constraints và signature dưới đây được ghi lại theo đúng bản LeetCode.

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def shortestDistance(self, grid: List[List[int]]) -> int:

```

Mục tiêu của bài học:

1. Làm quen với **lưới** (grid) trong Python từ con số 0: chỉ số `(hàng, cột)`, đi **4 hướng** bằng mảng `DIRS`, kiểm tra **biên** và cái bẫy "chỉ số âm".
2. Học **hàng đợi** (queue) bằng `collections.deque` và thuật toán **BFS** (Breadth-First Search — duyệt theo chiều rộng); hiểu **vì sao BFS cho đúng khoảng cách đường đi ngắn nhất** trên lưới không trọng số (mỗi bước tốn 1).
3. Hiểu vì sao bài này **không dùng được khoảng cách Manhattan** `|Δr| + |Δc|` (vật cản và tòa nhà chắn đường), và liên hệ với bài anh em **#296 Best Meeting Point** — nơi Manhattan lại đúng.
4. Đi từ cách trực giác nhất đến cách tối ưu:
   - **Cách 1** — BFS từ **mỗi ô trống** tới mọi tòa nhà (brute force, làm "trọng tài" để kiểm tra).
   - **Cách 2 (cách chính)** — BFS từ **mỗi tòa nhà**, cộng dồn vào hai bảng `dist_sum` và `reach`. Hiểu **khi nào nên BFS từ tòa nhà, khi nào từ ô trống**.
   - **Cách 3** — dùng **chính `grid` làm dấu** (`0, -1, -2, ...`): tòa nhà sau chỉ đi vào những ô mà **mọi tòa nhà trước** đã tới được, và dừng sớm khi không còn ô nào.
   - **Cách 4** — **phát hiện sớm `-1`**: nếu một tòa nhà không "chạm" được hết các tòa nhà khác thì trả `-1` ngay.
5. Kiểm chứng chéo mọi cách trên lưới ngẫu nhiên có `random.seed`, tránh các lỗi kinh điển và nhận dạng pattern "BFS nhiều nguồn trên lưới".

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho một lưới `grid` kích thước `m x n` chỉ gồm các giá trị `0`, `1`, `2`:
>
> - mỗi số `0` là một ô **đất trống** (empty land) — bạn được **đi qua tự do**;
> - mỗi số `1` là một **tòa nhà** (building) — bạn **không được đi xuyên qua**;
> - mỗi số `2` là một **vật cản** (obstacle) — bạn **không được đi xuyên qua**.
>
> Bạn muốn xây **một ngôi nhà** trên một ô đất trống sao cho **tổng quãng đường đi** từ ngôi nhà đó tới **tất cả** các tòa nhà là **nhỏ nhất**. Bạn chỉ được di chuyển lên, xuống, trái, phải.
>
> Trả về tổng quãng đường nhỏ nhất đó. Nếu **không thể** xây một ngôi nhà như vậy theo các quy tắc trên, trả về `-1`.

**Example 1**

```text
Input: grid = [[1,0,2,0,1],[0,0,0,0,0],[0,0,1,0,0]]
Output: 7
Giải thích: Có 3 tòa nhà ở (0,0), (0,4), (2,2) và một vật cản ở (0,2).
Ô (1,2) là ô đất trống lý tưởng để xây nhà, vì tổng quãng đường 3 + 3 + 1 = 7 là nhỏ nhất.
Vậy trả về 7.
```

**Example 2**

```text
Input: grid = [[1,0]]
Output: 1
```

**Example 3**

```text
Input: grid = [[1]]
Output: -1
```

**Constraints**

- `m == grid.length`
- `n == grid[i].length`
- `1 <= m, n <= 50`
- `grid[i][j]` là `0`, `1` hoặc `2`.
- Lưới có **ít nhất một** tòa nhà.

**Một chi tiết lạ trong đề gốc.** Bản đề trên LeetCode còn một câu (chép từ bài #296): *"The distance is calculated using Manhattan Distance, where distance(p1, p2) = |p2.x - p1.x| + |p2.y - p1.y|"*. Nhưng vì **không được đi xuyên tòa nhà và vật cản**, quãng đường thật phải **đi vòng** — các test của LeetCode chấm theo **số bước đi ngắn nhất trên lưới** (độ dài đường đi bộ), và mọi lời giải chuẩn đều dùng BFS. Mục 3 sẽ cho thấy bằng ví dụ cụ thể vì sao công thức Manhattan cho kết quả **sai**.

### Ký hiệu dùng suốt notebook

- `m` = số hàng, `n` = số cột, `N = m · n` = tổng số ô (tối đa `50 · 50 = 2500`).
- `B` = số tòa nhà, `E` = số ô đất trống (`B + E + số vật cản = N`).
- Ô ở hàng `r`, cột `c` viết là `(r, c)`.
- `dist(X, Y)` = **số bước ít nhất** để đi từ ô `X` tới ô `Y`, mỗi bước sang một ô kề cạnh, các ô **ở giữa** đường đi đều phải là đất trống. Đây là **khoảng cách đi bộ** (walking distance), **không phải** Manhattan.

### 1.1 Hiểu đề bằng hình

Ta vẽ lưới bằng ký hiệu cho dễ nhìn: `B` = tòa nhà (`1`), `#` = vật cản (`2`), `.` = đất trống (`0`), `H` = ngôi nhà ta định xây.

Example 1, xây nhà ở `H = (1,2)`:

```text
       c0 c1 c2 c3 c4
r0:     B  .  #  .  B
r1:     .  .  H  .  .
r2:     .  .  B  .  .
```

Ba đường đi ngắn nhất từ `H` tới ba tòa nhà:

| Tới tòa nhà | Một đường đi ngắn nhất | Số bước |
|---|---|:---:|
| `(0,0)` | `(1,2) → (1,1) → (1,0) → (0,0)` | 3 |
| `(0,4)` | `(1,2) → (1,3) → (1,4) → (0,4)` | 3 |
| `(2,2)` | `(1,2) → (2,2)` | 1 |
| **Tổng** | | **7** |

Vài suy luận nhỏ mà rất quan trọng:

- **Tòa nhà là điểm đến, không phải đường đi.** Bước cuối cùng được "bước vào" tòa nhà, nhưng không được đi **xuyên** qua một tòa nhà để tới chỗ khác. Ví dụ từ `(2,1)` muốn tới `(2,3)` thì không được đi thẳng qua `(2,2)` vì đó là tòa nhà.
- **Vật cản `#` chặn đường**: từ `(0,1)` tới `(0,3)` không thể đi thẳng qua `(0,2)`, phải vòng xuống hàng 1.
- Nhà chỉ được xây trên ô **đất trống** (`0`), không xây trên tòa nhà hay vật cản.
- Ngôi nhà phải **tới được mọi tòa nhà**. Nếu không có ô trống nào như vậy → `-1`. Example 3 `[[1]]` không có ô trống nào nên trả `-1`.

Cell dưới định nghĩa 3 ví dụ của đề, một hàm vẽ lưới ASCII, và một hàm **kiểm tra một đường đi có hợp lệ không** (kề cạnh, ô ở giữa phải là đất trống) — dùng nó để xác nhận bảng trên.

In [1]:
from typing import List
from collections import deque
import random
import time

EMPTY, BUILDING, OBSTACLE = 0, 1, 2
SYMBOL = {EMPTY: ".", BUILDING: "B", OBSTACLE: "#"}

ex1 = [[1, 0, 2, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]]
ex2 = [[1, 0]]
ex3 = [[1]]


def show_grid(grid, title=None, marks=None):
    # Vẽ lưới bằng ký hiệu; marks = {(r, c): "H"} để đánh dấu thêm ô đặc biệt.
    if title:
        print(title)
    n = len(grid[0])
    print("    " + " ".join(f"c{c}" for c in range(n)))
    for r, row in enumerate(grid):
        cells = []
        for c, v in enumerate(row):
            ch = marks[(r, c)] if marks and (r, c) in marks else SYMBOL[v]
            cells.append(f"{ch:>2}")
        print(f"r{r}: " + " ".join(cells))


def walk_length(grid, path):
    # path: danh sách ô từ nhà tới tòa nhà. Trả về số bước nếu hợp lệ, None nếu sai luật.
    for (r1, c1), (r2, c2) in zip(path, path[1:]):
        if abs(r1 - r2) + abs(c1 - c2) != 1:          # phải kề cạnh
            return None
    for r, c in path[1:-1]:                           # các ô Ở GIỮA phải là đất trống
        if grid[r][c] != EMPTY:
            return None
    return len(path) - 1


show_grid(ex1, "Example 1, xây nhà ở H = (1,2):", marks={(1, 2): "H"})
print("tới (0,0):", walk_length(ex1, [(1, 2), (1, 1), (1, 0), (0, 0)]))
print("tới (0,4):", walk_length(ex1, [(1, 2), (1, 3), (1, 4), (0, 4)]))
print("tới (2,2):", walk_length(ex1, [(1, 2), (2, 2)]))
print("đi xuyên vật cản (0,2):", walk_length(ex1, [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4)]))
print("đi xuyên tòa nhà (2,2):", walk_length(ex1, [(2, 1), (2, 2), (2, 3)]))

Example 1, xây nhà ở H = (1,2):
    c0 c1 c2 c3 c4
r0:  B  .  #  .  B
r1:  .  .  H  .  .
r2:  .  .  B  .  .
tới (0,0): 3
tới (0,4): 3
tới (2,2): 1
đi xuyên vật cản (0,2): None
đi xuyên tòa nhà (2,2): None


## 2. Nền tảng

### 2.1 Lưới là "list của các list"

Python không có kiểu "ma trận" riêng; ta dùng **list chứa các list**. Mỗi list con là **một hàng**.

```text
          c=0 c=1 c=2 c=3 c=4
r=0   [    1,  0,  2,  0,  1 ]   <- grid[0]
r=1   [    0,  0,  0,  0,  0 ]   <- grid[1]
r=2   [    0,  0,  1,  0,  0 ]   <- grid[2]
```

- `grid[r]` là cả hàng `r`; `grid[r][c]` là ô hàng `r`, cột `c` — **hàng trước, cột sau**.
- `m = len(grid)` là số hàng; `n = len(grid[0])` là số cột.
- `r` tăng là đi **xuống**, `c` tăng là đi sang **phải**.

Ta cũng hay cần **tạo một bảng mới cùng kích thước** (ví dụ bảng khoảng cách). Cách đúng là `[[0] * n for _ in range(m)]` — mỗi hàng là một list **riêng**. Cách sai `[[0] * n] * m` tạo ra `m` **tham chiếu tới cùng một hàng**: sửa một ô là cả cột đổi theo.

In [2]:
grid = ex1
m, n = len(grid), len(grid[0])
print("m =", m, ", n =", n)
print("grid[0]    =", grid[0])
print("grid[2][2] =", grid[2][2], "-> tòa nhà")
print("grid[0][2] =", grid[0][2], "-> vật cản")

buildings = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == BUILDING]
empties = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == EMPTY]
print("các tòa nhà:", buildings, "-> B =", len(buildings))
print("số ô trống E =", len(empties))

good = [[0] * 3 for _ in range(2)]
bad = [[0] * 3] * 2
good[0][0] = 9
bad[0][0] = 9
print("tạo đúng:", good)
print("tạo sai :", bad, " <- hàng 1 cũng bị đổi!")

m = 3 , n = 5
grid[0]    = [1, 0, 2, 0, 1]
grid[2][2] = 1 -> tòa nhà
grid[0][2] = 2 -> vật cản
các tòa nhà: [(0, 0), (0, 4), (2, 2)] -> B = 3
số ô trống E = 11
tạo đúng: [[9, 0, 0], [0, 0, 0]]
tạo sai : [[9, 0, 0], [9, 0, 0]]  <- hàng 1 cũng bị đổi!


### 2.2 Bốn hướng đi, kiểm tra biên và cái bẫy "chỉ số âm"

Từ ô `(r, c)`, bốn ô kề cạnh là:

| Hướng | `dr` | `dc` | Ô mới |
|---|:---:|:---:|---|
| lên | `-1` | `0` | `(r-1, c)` |
| xuống | `+1` | `0` | `(r+1, c)` |
| trái | `0` | `-1` | `(r, c-1)` |
| phải | `0` | `+1` | `(r, c+1)` |

Gom 4 cặp `(dr, dc)` vào list `DIRS` rồi lặp, thay vì viết 4 khối `if` giống nhau:

```python
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải
for dr, dc in DIRS:
    nr, nc = r + dr, c + dc                  # "new row", "new column"
```

Ô mới chỉ hợp lệ khi nằm **trong lưới**: `0 <= nr < m and 0 <= nc < n` (Python cho viết so sánh "kẹp hai đầu").

**Bẫy riêng của Python**: chỉ số âm **không báo lỗi**! `grid[-1]` là hàng **cuối cùng**. Nếu quên vế `0 <= nr`, đứng ở hàng 0 mà đi "lên" sẽ lặng lẽ đọc hàng cuối — một nước đi "xuyên tường" không tồn tại.

Bài này còn thêm một lớp lọc: ô mới phải là **đất trống** thì mới được **đi tiếp** từ nó. Cell dưới in các hàng xóm và phân loại chúng.

In [3]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # lên, xuống, trái, phải


def neighbors(grid, r, c):
    # Các ô kề cạnh (r, c) nằm TRONG lưới.
    m, n = len(grid), len(grid[0])
    result = []
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < m and 0 <= nc < n:
            result.append((nr, nc))
    return result


kind = {EMPTY: "trống -> đi tiếp được", BUILDING: "tòa nhà -> chỉ được bước VÀO", OBSTACLE: "vật cản -> cấm"}
for cell in [(0, 1), (1, 2), (0, 0)]:
    print(f"hàng xóm của {cell}:")
    for nr, nc in neighbors(ex1, *cell):
        print(f"    ({nr},{nc}) = {ex1[nr][nc]}  {kind[ex1[nr][nc]]}")
print("bẫy chỉ số âm: ex1[-1][2] =", ex1[-1][2], "-> đó là ô (2,2), không phải ô 'phía trên' (0,2)")

hàng xóm của (0, 1):
    (1,1) = 0  trống -> đi tiếp được
    (0,0) = 1  tòa nhà -> chỉ được bước VÀO
    (0,2) = 2  vật cản -> cấm
hàng xóm của (1, 2):
    (0,2) = 2  vật cản -> cấm
    (2,2) = 1  tòa nhà -> chỉ được bước VÀO
    (1,1) = 0  trống -> đi tiếp được
    (1,3) = 0  trống -> đi tiếp được
hàng xóm của (0, 0):
    (1,0) = 0  trống -> đi tiếp được
    (0,1) = 0  trống -> đi tiếp được
bẫy chỉ số âm: ex1[-1][2] = 1 -> đó là ô (2,2), không phải ô 'phía trên' (0,2)


### 2.3 Hàng đợi (queue) và `collections.deque`

**Hàng đợi** (queue) hoạt động như xếp hàng mua vé: ai đến **trước** được phục vụ **trước** — gọi là **FIFO** (First In, First Out).

- **Thêm vào cuối** hàng: `q.append(x)`.
- **Lấy ra ở đầu** hàng: `q.popleft()`.

Trong Python, dùng `deque` (đọc là "đéc", viết tắt của *double-ended queue*) trong module `collections`. Vì sao không dùng `list`? Vì `list.pop(0)` phải **dời toàn bộ** các phần tử còn lại lên một chỗ → tốn `O(độ dài list)` mỗi lần. `deque.popleft()` chỉ tốn `O(1)`.

Ví dụ mini: 3 người `An, Bình, Chi` xếp hàng, phục vụ một người rồi `Dũng` tới.

In [4]:
q = deque()
q.append("An")
q.append("Bình")
q.append("Chi")
print("hàng đợi      :", list(q))
print("phục vụ       :", q.popleft())      # An đến trước -> ra trước
q.append("Dũng")
print("sau khi Dũng tới:", list(q))
print("còn bao nhiêu :", len(q), "| rỗng chưa?", not q)

# So sánh tốc độ: lấy hết 20 000 phần tử từ đầu
K = 20_000
lst = list(range(K))
t0 = time.perf_counter()
while lst:
    lst.pop(0)
t_list = time.perf_counter() - t0
dq = deque(range(K))
t0 = time.perf_counter()
while dq:
    dq.popleft()
t_deque = time.perf_counter() - t0
print(f"list.pop(0): {t_list * 1000:.1f} ms | deque.popleft(): {t_deque * 1000:.1f} ms")

hàng đợi      : ['An', 'Bình', 'Chi']
phục vụ       : An
sau khi Dũng tới: ['Bình', 'Chi', 'Dũng']
còn bao nhiêu : 3 | rỗng chưa? False
list.pop(0): 31.3 ms | deque.popleft(): 0.8 ms


### 2.4 BFS trên lưới: "vết dầu loang" từ một ô

**BFS** (Breadth-First Search — duyệt theo chiều rộng) giống như **giọt dầu loang** hoặc **gợn sóng** khi ném hòn đá xuống hồ: đầu tiên chạm các ô cách 1 bước, rồi các ô cách 2 bước, rồi 3 bước...

Cách làm với hàng đợi:

1. Tạo bảng `dist` cùng kích thước lưới, toàn `-1` (nghĩa là **chưa tới**). Đặt `dist[nguồn] = 0` và đưa nguồn vào hàng đợi.
2. Lặp: lấy ô `(r, c)` ở **đầu** hàng đợi. Với mỗi hàng xóm `(nr, nc)` **trong lưới**, là **đất trống** và **chưa tới** (`dist == -1`):
   - `dist[nr][nc] = dist[r][c] + 1`,
   - đưa `(nr, nc)` vào **cuối** hàng đợi.
3. Hàng đợi rỗng → xong. Ô nào vẫn `-1` là **không tới được**.

Điểm then chốt: **đánh dấu ngay lúc đưa vào hàng đợi** (gán `dist` khi `append`), không đợi lúc lấy ra — nhờ vậy mỗi ô vào hàng đợi **đúng một lần**.

Ta thử trên một "mê cung" nhỏ: tòa nhà ở góc trên trái, một bức tường vật cản ở hàng 1.

```text
       c0 c1 c2 c3
r0:     B  .  .  .
r1:     #  #  .  #
r2:     .  .  .  .
```

Dự đoán: muốn từ `B` xuống ô `(2,0)` phải đi `→ → ↓ ↓ ← ←`, tức **6 bước** — dù về mặt "đường chim bay" `(2,0)` chỉ cách `(0,0)` có 2 ô.

In [5]:
maze = [[1, 0, 0, 0],
        [2, 2, 0, 2],
        [0, 0, 0, 0]]


def bfs_distances(grid, sr, sc):
    # BFS từ ô nguồn (sr, sc); CHỈ được đi qua ô trống.
    # Trả về (dist, touched):
    #   dist[r][c]  = số bước ít nhất tới ô (r, c), -1 nếu không tới được
    #   touched     = {ô tòa nhà: số bước tới nó} -- tòa nhà được "chạm" nhưng KHÔNG đi xuyên qua
    m, n = len(grid), len(grid[0])
    dist = [[-1] * n for _ in range(m)]
    dist[sr][sc] = 0
    touched = {}
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(grid, r, c):
            if dist[nr][nc] != -1 or (nr, nc) == (sr, sc):
                continue
            if grid[nr][nc] == EMPTY:
                dist[nr][nc] = dist[r][c] + 1       # đánh dấu NGAY khi đưa vào hàng đợi
                q.append((nr, nc))
            elif grid[nr][nc] == BUILDING and (nr, nc) not in touched:
                touched[(nr, nc)] = dist[r][c] + 1  # bước VÀO tòa nhà, nhưng không đi tiếp
    return dist, touched


def show_values(grid, values, title=None, width=3):
    # In bảng số trên lưới: tòa nhà = B, vật cản = #, ô chưa tới (-1) = '.'
    if title:
        print(title)
    for r, row in enumerate(grid):
        out = []
        for c, v in enumerate(row):
            if v != EMPTY:
                out.append(SYMBOL[v])
            else:
                out.append("." if values[r][c] == -1 else str(values[r][c]))
        print(" ".join(f"{x:>{width}}" for x in out))


dist, touched = bfs_distances(maze, 0, 0)
show_values(maze, dist, "Khoảng cách đi bộ từ tòa nhà (0,0):")
print("dist tới (2,0) =", dist[2][0], " (đường chim bay chỉ là", abs(2 - 0) + abs(0 - 0), ")")

Khoảng cách đi bộ từ tòa nhà (0,0):
  B   1   2   3
  #   #   3   #
  6   5   4   5
dist tới (2,0) = 6  (đường chim bay chỉ là 2 )


### 2.5 Vì sao BFS cho đúng **đường đi ngắn nhất**? Trace từng bước

Trên lưới **không trọng số** (mỗi bước tốn đúng 1), BFS xử lý các ô theo **từng lớp**: lớp 0 (nguồn), rồi toàn bộ lớp 1, rồi toàn bộ lớp 2...

**Invariant (bất biến) của hàng đợi**: tại mọi thời điểm, các ô trong hàng đợi có `dist` **không giảm** từ đầu tới cuối, và chênh nhau **tối đa 1** (dạng `k, k, ..., k, k+1, ..., k+1`). Lý do: ta lấy ra một ô lớp `k` thì chỉ đẩy vào cuối các ô lớp `k+1`.

Hệ quả: **lần đầu tiên** một ô được phát hiện, nó được phát hiện từ một ô ở lớp nhỏ nhất có thể → `dist` gán lúc đó chính là số bước ít nhất. Sau đó ô bị đánh dấu nên không bao giờ bị gán lại bằng một giá trị lớn hơn.

(So sánh: DFS — đi sâu trước — có thể tới một ô bằng đường **vòng** trước, nên **không** cho khoảng cách ngắn nhất.)

Hàm `trace_bfs` dưới in, ở mỗi lần lấy ra: ô đang xét, `dist` của nó, những ô mới được đẩy vào, và nội dung hàng đợi sau đó. Hãy để ý cột `dist` trong hàng đợi luôn có dạng `k ... k+1`.

In [6]:
def trace_bfs(grid, sr, sc):
    m, n = len(grid), len(grid[0])
    dist = [[-1] * n for _ in range(m)]
    dist[sr][sc] = 0
    q = deque([(sr, sc)])
    step = 0
    while q:
        r, c = q.popleft()
        step += 1
        pushed = []
        for nr, nc in neighbors(grid, r, c):
            if grid[nr][nc] == EMPTY and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
                pushed.append(f"({nr},{nc})")
        queue_view = " ".join(f"({a},{b}):{dist[a][b]}" for a, b in q)
        print(f"bước {step}: lấy ({r},{c}) dist={dist[r][c]} | đẩy {', '.join(pushed) or '-':<14} | hàng đợi: {queue_view or 'rỗng'}")
    return dist


show_grid(maze, "Mê cung:")
_ = trace_bfs(maze, 0, 0)

Mê cung:
    c0 c1 c2 c3
r0:  B  .  .  .
r1:  #  #  .  #
r2:  .  .  .  .
bước 1: lấy (0,0) dist=0 | đẩy (0,1)          | hàng đợi: (0,1):1
bước 2: lấy (0,1) dist=1 | đẩy (0,2)          | hàng đợi: (0,2):2
bước 3: lấy (0,2) dist=2 | đẩy (1,2), (0,3)   | hàng đợi: (1,2):3 (0,3):3
bước 4: lấy (1,2) dist=3 | đẩy (2,2)          | hàng đợi: (0,3):3 (2,2):4
bước 5: lấy (0,3) dist=3 | đẩy -              | hàng đợi: (2,2):4
bước 6: lấy (2,2) dist=4 | đẩy (2,1), (2,3)   | hàng đợi: (2,1):5 (2,3):5
bước 7: lấy (2,1) dist=5 | đẩy (2,0)          | hàng đợi: (2,3):5 (2,0):6
bước 8: lấy (2,3) dist=5 | đẩy -              | hàng đợi: (2,0):6
bước 9: lấy (2,0) dist=6 | đẩy -              | hàng đợi: rỗng


### 2.6 BFS **xuất phát từ một tòa nhà**: chạm vào tòa nhà khác nhưng không đi xuyên

Bài này có 3 loại ô, nên khi BFS ta phải phân biệt rõ:

| Hàng xóm là | Xử lý |
|---|---|
| đất trống `0`, chưa tới | gán `dist`, đưa vào hàng đợi (đi tiếp từ đó) |
| tòa nhà `1` | **ghi nhận đã chạm** (biết được khoảng cách tới nó), nhưng **không** đưa vào hàng đợi |
| vật cản `2` hoặc ngoài lưới | bỏ qua |

Hàm `bfs_distances` ở mục 2.4 đã làm đúng như vậy: dict `touched` lưu các tòa nhà chạm được cùng số bước.

Thử BFS từ tòa nhà `(0,0)` của Example 1. Chú ý ô `(0,3)` cách `(0,0)` chỉ 3 cột nhưng phải đi **5 bước** (vòng xuống hàng 1 để tránh vật cản `(0,2)`); ô `(2,3)` cũng phải vòng qua tòa nhà `(2,2)`.

In [7]:
dist, touched = bfs_distances(ex1, 0, 0)
show_grid(ex1, "Example 1:")
show_values(ex1, dist, "Khoảng cách đi bộ từ tòa nhà (0,0) tới từng ô trống:")
print("các tòa nhà khác chạm được:", touched)

Example 1:
    c0 c1 c2 c3 c4
r0:  B  .  #  .  B
r1:  .  .  .  .  .
r2:  .  .  B  .  .
Khoảng cách đi bộ từ tòa nhà (0,0) tới từng ô trống:
  B   1   #   5   B
  1   2   3   4   5
  2   3   B   5   6
các tòa nhà khác chạm được: {(2, 2): 4, (0, 4): 6}


## 3. Vì sao **không** dùng được khoảng cách Manhattan?

### 3.1 Manhattan là gì, và bài anh em #296

**Khoảng cách Manhattan** giữa hai ô `(r1, c1)` và `(r2, c2)` là `|r1 - r2| + |c1 - c2|` — số bước nếu đi **lưới trống trơn**, không có gì chắn đường (giống đi bộ trong khu phố ô bàn cờ ở Manhattan).

Ở bài **#296 Best Meeting Point** (Hard, Premium): lưới chỉ có `0` và `1` (nhà của bạn bè), ai cũng **đi xuyên mọi ô** được, điểm hẹn có thể là **bất kỳ ô nào**. Khi đó khoảng cách đúng **bằng** Manhattan, và có mẹo rất đẹp: vì `|Δr| + |Δc|` tách được thành phần hàng và phần cột, ta chọn **hàng trung vị** (median) của các hàng và **cột trung vị** của các cột → `O(m·n)`, không cần BFS. (Bài tập 5 sẽ cho bạn tự cài #296.)

Bài #317 khác ở **3 chỗ**, và mỗi chỗ đều làm Manhattan sai:

1. **Vật cản `#`** buộc phải đi vòng → quãng đường thật **dài hơn** Manhattan.
2. **Tòa nhà cũng chắn đường** (không đi xuyên) → kể cả lưới không có `#` vẫn phải đi vòng, thậm chí **không tới được**.
3. **Có thể không tới được** một số tòa nhà → Manhattan vẫn cho ra một con số hữu hạn, trong khi đáp án thật là `-1` (hoặc ô đó không được chọn).

Cell dưới so sánh Manhattan với khoảng cách đi bộ (BFS) từ tòa nhà `(0,0)` trong mê cung ở mục 2.4.

In [8]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


dist, _ = bfs_distances(maze, 0, 0)
show_grid(maze, "Mê cung:")
print("ô     | Manhattan | đi bộ (BFS)")
for r in range(len(maze)):
    for c in range(len(maze[0])):
        if maze[r][c] == EMPTY:
            flag = "  <- lệch!" if manhattan((0, 0), (r, c)) != dist[r][c] else ""
            print(f"({r},{c}) | {manhattan((0, 0), (r, c)):^9} | {dist[r][c]:^11}{flag}")

Mê cung:
    c0 c1 c2 c3
r0:  B  .  .  .
r1:  #  #  .  #
r2:  .  .  .  .
ô     | Manhattan | đi bộ (BFS)
(0,1) |     1     |      1     
(0,2) |     2     |      2     
(0,3) |     3     |      3     
(1,2) |     3     |      3     
(2,0) |     2     |      6       <- lệch!
(2,1) |     3     |      5       <- lệch!
(2,2) |     4     |      4     
(2,3) |     5     |      5     


### 3.2 Ba phản ví dụ cho "lời giải Manhattan"

Một người mới có thể viết: *"với mỗi ô trống, cộng Manhattan tới mọi tòa nhà, lấy min"*. Ta gọi nó là `sd_manhattan_wrong` và đưa vào 3 lưới:

**(a) Bức tường** — `wall`:

```text
       c0 c1 c2 c3
r0:     B  .  .  .
r1:     #  #  #  .
r2:     B  .  .  .
```

Manhattan nói ô `(0,1)` chỉ cách hai tòa nhà `1 + 3 = 4`. Thật ra từ `(0,1)` tới tòa nhà `(2,0)` phải đi vòng chữ U: `(0,2),(0,3),(1,3),(2,3),(2,2),(2,1),(2,0)` = **7 bước**, tổng `1 + 7 = 8`. Mọi ô trên chữ U đều có tổng **8** (đường nối hai tòa nhà dài đúng 8) → đáp án thật **8**.

**(b) Tòa nhà chắn đường** — `[[1, 0, 1, 0, 1]]`: ô `(0,1)` muốn tới tòa nhà `(0,4)` thì phải đi xuyên tòa nhà `(0,2)` — cấm. Tương tự `(0,3)` không tới được `(0,0)`. Đáp án thật **`-1`**, Manhattan lại trả `5`. (Còn nếu đây là bài #296 — được đi xuyên và hẹn ở bất kỳ ô nào — đáp án là `4`, hẹn ở chính giữa.)

**(c) Ô bị nhốt** — `[[1, 1, 0]]`: ô trống `(0,2)` chỉ chạm được tòa nhà `(0,1)`; tòa nhà `(0,0)` bị `(0,1)` chắn. Đáp án thật **`-1`**, Manhattan trả `3`.

(Các đáp án thật ở đây sẽ được kiểm chứng lại bằng assert ở mục 7 khi đã có lời giải đúng.)

In [9]:
def sd_manhattan_wrong(grid):
    # SAI: coi như đi xuyên được mọi ô
    m, n = len(grid), len(grid[0])
    bs = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == BUILDING]
    totals = [sum(manhattan((r, c), b) for b in bs)
              for r in range(m) for c in range(n) if grid[r][c] == EMPTY]
    return min(totals) if totals else -1


wall = [[1, 0, 0, 0],
        [2, 2, 2, 0],
        [1, 0, 0, 0]]
cases = [("(a) bức tường", wall, 8), ("(b) tòa nhà chắn", [[1, 0, 1, 0, 1]], -1), ("(c) ô bị nhốt", [[1, 1, 0]], -1)]
for name, g, truth in cases:
    print(f"{name:<18} Manhattan = {sd_manhattan_wrong(g):>2} | đáp án thật = {truth:>2}")
print("Example 1 (tình cờ trùng):", sd_manhattan_wrong(ex1), "vs 7")

(a) bức tường      Manhattan =  4 | đáp án thật =  8
(b) tòa nhà chắn   Manhattan =  5 | đáp án thật = -1
(c) ô bị nhốt      Manhattan =  3 | đáp án thật = -1
Example 1 (tình cờ trùng): 7 vs 7


**Bài học:** Example 1 của đề tình cờ cho Manhattan **trùng** đáp án (7), nên lời giải sai vẫn qua được ví dụ — đây đúng là loại bẫy "pass ví dụ nhưng fail khi submit". Khi lưới có **thứ chắn đường**, khoảng cách phải tính bằng **BFS**.

**Mẹo nhận dạng:** "được đi xuyên mọi ô" → Manhattan + trung vị (#296). "Có ô cấm đi qua" → BFS (#317).

## 4. Cách 1 — BFS từ **mỗi ô trống** (brute force, làm "trọng tài")

### 4.1 Ý tưởng tự nhiên nhất

Người mới sẽ nghĩ đúng như cách đề phát biểu: *"thử xây nhà ở từng ô trống, tính tổng quãng đường tới mọi tòa nhà, giữ lại tổng nhỏ nhất"*.

Với mỗi ô trống `X`:

1. Chạy BFS từ `X` (hàm `bfs_distances` ở mục 2.4) → biết `touched` = các tòa nhà chạm được và số bước tới mỗi cái.
2. Nếu `len(touched) < B` → `X` **không** tới được mọi tòa nhà → loại.
3. Ngược lại, `tổng(X) = sum(touched.values())`; cập nhật `best = min(best, tổng(X))`.

Cuối cùng, nếu không ô nào hợp lệ (`best` vẫn là vô cùng `float("inf")`) → trả `-1`.

Cách này **chậm** nhưng **cực kỳ dễ thấy là đúng** — nó làm đúng nguyên văn định nghĩa của đề. Vì vậy ta giữ nó làm **trọng tài** (reference) để kiểm tra các cách nhanh hơn trên lưới nhỏ.

In [10]:
def sd_bruteforce(grid):
    m, n = len(grid), len(grid[0])
    total_buildings = sum(row.count(BUILDING) for row in grid)
    best = float("inf")
    for r in range(m):
        for c in range(n):
            if grid[r][c] != EMPTY:
                continue
            _, touched = bfs_distances(grid, r, c)       # BFS từ ô trống (r, c)
            if len(touched) == total_buildings:          # tới được MỌI tòa nhà?
                best = min(best, sum(touched.values()))
    return -1 if best == float("inf") else best


print("Example 1:", sd_bruteforce(ex1))   # 7
print("Example 2:", sd_bruteforce(ex2))   # 1
print("Example 3:", sd_bruteforce(ex3))   # -1
print("wall     :", sd_bruteforce(wall))  # 8

Example 1: 7
Example 2: 1
Example 3: -1
wall     : 8


### 4.2 Nhìn tổng của **từng ô** trên Example 1

Để "thấy" bài toán, in ra tổng quãng đường nếu xây nhà ở từng ô trống (`--` nếu ô đó không tới được mọi tòa nhà). Đáp án là số nhỏ nhất trên bảng.

Dự đoán trước: ô `(1,2)` nằm "giữa" ba tòa nhà và ngay trên tòa nhà `(2,2)` → nhiều khả năng nhỏ nhất (7). Các góc dưới `(2,0)`, `(2,4)` xa cả hai tòa nhà phía trên lẫn phải vòng qua `(2,2)` → lớn nhất.

In [11]:
def totals_table(grid):
    m, n = len(grid), len(grid[0])
    total_buildings = sum(row.count(BUILDING) for row in grid)
    for r in range(m):
        out = []
        for c in range(n):
            if grid[r][c] != EMPTY:
                out.append(SYMBOL[grid[r][c]])
                continue
            _, touched = bfs_distances(grid, r, c)
            out.append(str(sum(touched.values())) if len(touched) == total_buildings else "--")
        print(" ".join(f"{x:>3}" for x in out))


totals_table(ex1)

  B   9   #   9   B
  9   8   7   8   9
 10   9   B   9  10


### 4.3 Brute force tốn bao nhiêu? Vướng ở đâu?

- Có `E` ô trống, mỗi ô chạy một BFS. Mỗi BFS duyệt tối đa mọi ô trống (mỗi ô xét 4 hàng xóm) và còn cấp phát bảng `dist` `m × n` → `O(m·n)`.
- Tổng: **`O(E · m · n)`**, mà `E` có thể gần bằng `m·n` → **`O((m·n)²)`**. Với `50 × 50`: khoảng `2500 × 2500 ≈ 6 triệu` lượt xét ô, nhân thêm hằng số 4 hướng — trong Python mất vài giây. Chưa kể LeetCode có nhiều test.

**Công việc nào đang bị lặp?** Nhìn kỹ: mỗi BFS từ một ô trống đều "loang" ra **cả vùng** chỉ để đo khoảng cách tới **một vài** tòa nhà. Nếu lưới có 2500 ô trống mà chỉ 3 tòa nhà, ta chạy 2500 lần BFS — trong khi thông tin thật sự cần chỉ là **3 bảng khoảng cách** (mỗi tòa nhà một bảng).

**Mấu chốt cải tiến:** khoảng cách đi bộ **đối xứng**: `dist(X, tòa nhà) = dist(tòa nhà, X)`. Vậy thay vì BFS từ mỗi ô trống, hãy **BFS từ mỗi tòa nhà** một lần — mỗi lần cho ta khoảng cách tới **tất cả** ô trống cùng lúc.

## 5. Bước đột phá: đảo chiều BFS + hai bảng cộng dồn

### 5.1 Khoảng cách đi bộ đối xứng

Đi từ `X` tới `Y` theo một đường hợp lệ thì **đi ngược** đường đó là đường hợp lệ từ `Y` về `X`, cùng số bước (các ô ở giữa vẫn là đất trống, vẫn kề cạnh nhau). Nên đường ngắn nhất hai chiều dài **bằng nhau**: lưới là một **đồ thị vô hướng** (undirected graph).

Kiểm chứng trên mê cung: BFS **từ ô trống** `(2,0)` chạm tòa nhà `(0,0)` sau mấy bước? Và BFS **từ tòa nhà** `(0,0)` tới `(2,0)` mấy bước?

In [12]:
dist_from_building, _ = bfs_distances(maze, 0, 0)
_, touched_from_empty = bfs_distances(maze, 2, 0)
print("BFS từ tòa nhà (0,0), dist tới (2,0)   :", dist_from_building[2][0])
print("BFS từ ô trống (2,0), chạm tòa nhà (0,0):", touched_from_empty[(0, 0)])

# Kiểm tra trên MỌI cặp (ô trống, tòa nhà) của Example 1
ok = True
for br, bc in [(0, 0), (0, 4), (2, 2)]:
    d_b, _ = bfs_distances(ex1, br, bc)
    for r in range(3):
        for c in range(5):
            if ex1[r][c] == EMPTY:
                _, t = bfs_distances(ex1, r, c)
                ok &= (t.get((br, bc), -1) == d_b[r][c])
print("đối xứng trên mọi cặp của Example 1:", ok)

BFS từ tòa nhà (0,0), dist tới (2,0)   : 6
BFS từ ô trống (2,0), chạm tòa nhà (0,0): 6
đối xứng trên mọi cặp của Example 1: True


### 5.2 Hai bảng cộng dồn: `dist_sum` và `reach`

BFS từ tòa nhà thứ `k` cho ta bảng `dist_k`. Tổng quãng đường nếu xây nhà ở ô `X` chính là

```text
tổng(X) = dist_1[X] + dist_2[X] + ... + dist_B[X]
```

Ta **không cần giữ** `B` bảng: chỉ cần một bảng `dist_sum` và **cộng dồn** ngay trong lúc BFS: mỗi lần BFS từ tòa nhà `k` tới ô `X` với `d` bước, làm `dist_sum[X] += d`.

Nhưng nếu tòa nhà `k` **không tới được** `X` thì sao? Khi đó `dist_sum[X]` chỉ **thiếu đi** một số hạng — nó trông **nhỏ hơn** thật, và có thể "thắng" sai! Vì vậy cần bảng thứ hai:

- `reach[X]` = **số tòa nhà đã tới được** `X`. Mỗi lần BFS từ một tòa nhà tới `X`: `reach[X] += 1`.
- Cuối cùng chỉ xét các ô có `reach[X] == B` (mọi tòa nhà đều tới được).

Ví dụ mini để thấy vì sao `reach` bắt buộc — lưới `pocket`:

```text
       c0 c1 c2 c3 c4
r0:     .  B  .  .  .
r1:     #  .  .  #  .
r2:     .  .  .  #  B
r3:     B  #  .  .  .
```

Ô `(0,0)` là một "túi" bị kẹp giữa tòa nhà `(0,1)` và vật cản `(1,0)`: **chỉ** tòa nhà `(0,1)` bước được vào nó (1 bước). Hai tòa nhà còn lại không vào được vì muốn vào phải đi xuyên `(0,1)`. Nên `dist_sum[(0,0)] = 1` — nhỏ nhất bảng — nhưng `reach[(0,0)] = 1 < 3`, phải loại.

In [13]:
pocket = [[0, 1, 0, 0, 0],
          [2, 0, 0, 2, 0],
          [0, 0, 0, 2, 1],
          [1, 2, 0, 0, 0]]

m, n = len(pocket), len(pocket[0])
dist_sum = [[0] * n for _ in range(m)]
reach = [[0] * n for _ in range(m)]
for br in range(m):
    for bc in range(n):
        if pocket[br][bc] == BUILDING:
            d, _ = bfs_distances(pocket, br, bc)
            for r in range(m):
                for c in range(n):
                    if pocket[r][c] == EMPTY and d[r][c] != -1:
                        dist_sum[r][c] += d[r][c]
                        reach[r][c] += 1

show_values(pocket, dist_sum, "dist_sum:")
show_values(pocket, reach, "reach:")
print("dist_sum nhỏ nhất (KHÔNG xét reach):", min(dist_sum[r][c] for r in range(m) for c in range(n) if pocket[r][c] == EMPTY))
print("dist_sum nhỏ nhất với reach == 3    :", min(dist_sum[r][c] for r in range(m) for c in range(n) if pocket[r][c] == EMPTY and reach[r][c] == 3))

dist_sum:
  1   B  10  11  12
  #  10  11   #  13
 10   9  10   #   B
  B   #  11  12  13
reach:
  1   B   3   3   3
  #   3   3   #   3
  3   3   3   #   B
  B   #   3   3   3
dist_sum nhỏ nhất (KHÔNG xét reach): 1
dist_sum nhỏ nhất với reach == 3    : 9


## 6. Cách 2 (cách chính) — BFS từ mỗi tòa nhà, cộng dồn `dist_sum` và `reach`

### 6.1 Ghép lại thành một hàm hoàn chỉnh

```text
dist_sum = bảng 0, reach = bảng 0, B = 0
với mỗi tòa nhà (br, bc):
    B += 1
    BFS từ (br, bc), chỉ đi trên ô trống:
        khi lần đầu tới ô trống (nr, nc) với d bước:
            dist_sum[nr][nc] += d
            reach[nr][nc]    += 1
đáp án = min dist_sum[X] trên các ô trống X có reach[X] == B   (không có -> -1)
```

Chi tiết cài đặt:

- Mỗi BFS cần bảng `dist` **riêng** (toàn `-1`) để biết ô nào đã tới **trong lần BFS này**. Không dùng chung giữa các tòa nhà, vì tòa nhà sau phải được đi lại mọi ô.
- Hàm `min(..., default=-1)`: nếu dãy rỗng (không có ô hợp lệ) thì trả `default` thay vì báo lỗi `ValueError`. Gọn hơn việc dùng `float("inf")` rồi đổi sang `-1`.

In [14]:
def sd_from_buildings(grid):
    m, n = len(grid), len(grid[0])
    dist_sum = [[0] * n for _ in range(m)]   # tổng quãng đường từ mọi tòa nhà đã BFS
    reach = [[0] * n for _ in range(m)]      # số tòa nhà đã tới được ô này
    total_buildings = 0

    for br in range(m):
        for bc in range(n):
            if grid[br][bc] != BUILDING:
                continue
            total_buildings += 1
            dist = [[-1] * n for _ in range(m)]          # bảng riêng cho lần BFS này
            dist[br][bc] = 0
            q = deque([(br, bc)])
            while q:
                r, c = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == EMPTY and dist[nr][nc] == -1:
                        dist[nr][nc] = dist[r][c] + 1
                        dist_sum[nr][nc] += dist[nr][nc]
                        reach[nr][nc] += 1
                        q.append((nr, nc))

    return min((dist_sum[r][c] for r in range(m) for c in range(n)
                if grid[r][c] == EMPTY and reach[r][c] == total_buildings), default=-1)


for name, g in [("Example 1", ex1), ("Example 2", ex2), ("Example 3", ex3), ("wall", wall), ("pocket", pocket)]:
    print(f"{name:<10}: {sd_from_buildings(g):>2} | brute force: {sd_bruteforce(g):>2}")

Example 1 :  7 | brute force:  7
Example 2 :  1 | brute force:  1
Example 3 : -1 | brute force: -1
wall      :  8 | brute force:  8
pocket    :  9 | brute force:  9


### 6.2 Trace: bảng `dist_sum` sau từng tòa nhà (Example 1)

Ba tòa nhà được duyệt theo thứ tự quét hàng: `(0,0)`, `(0,4)`, `(2,2)`. Dưới đây là `dist_sum` sau mỗi lần BFS (`B` = tòa nhà, `#` = vật cản):

| Sau BFS từ | `dist_sum` hàng 0 | `dist_sum` hàng 1 | `dist_sum` hàng 2 |
|---|---|---|---|
| `(0,0)` | `B 1 # 5 B` | `1 2 3 4 5` | `2 3 B 5 6` |
| `(0,4)` | `B 6 # 6 B` | `6 6 6 6 6` | `8 8 B 8 8` |
| `(2,2)` | `B 9 # 9 B` | `9 8 7 8 9` | `10 9 B 9 10` |

Để ý vài ô cụ thể:

| Ô | từ `(0,0)` | từ `(0,4)` | từ `(2,2)` | Tổng |
|---|:---:|:---:|:---:|:---:|
| `(1,2)` | 3 | 3 | 1 | **7** ← nhỏ nhất |
| `(1,1)` | 2 | 4 | 2 | 8 |
| `(0,1)` | 1 | 5 | 3 | 9 — `(0,1)` gần `(0,0)` nhưng tới `(0,4)` phải vòng vật cản |
| `(2,0)` | 2 | 6 | 2 | 10 |

Ở Example 1, sau 3 lần BFS mọi ô trống đều có `reach = 3` (không ô nào bị nhốt), nên đáp án là `min dist_sum = 7` tại `(1,2)` — khớp bảng ở mục 4.2.

Hàm `trace_from_buildings` in, với mỗi tòa nhà, ba bảng đặt cạnh nhau: khoảng cách **từ tòa nhà này**, `dist_sum` **sau khi cộng**, và `reach`.

In [15]:
def trace_from_buildings(grid):
    m, n = len(grid), len(grid[0])
    dist_sum = [[0] * n for _ in range(m)]
    reach = [[0] * n for _ in range(m)]
    k = 0

    def row_str(r, table):
        out = []
        for c in range(n):
            if grid[r][c] != EMPTY:
                out.append(SYMBOL[grid[r][c]])
            else:
                out.append("." if table[r][c] == -1 else str(table[r][c]))
        return " ".join(f"{x:>2}" for x in out)

    for br in range(m):
        for bc in range(n):
            if grid[br][bc] != BUILDING:
                continue
            k += 1
            dist = [[-1] * n for _ in range(m)]
            dist[br][bc] = 0
            q = deque([(br, bc)])
            while q:
                r, c = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == EMPTY and dist[nr][nc] == -1:
                        dist[nr][nc] = dist[r][c] + 1
                        dist_sum[nr][nc] += dist[nr][nc]
                        reach[nr][nc] += 1
                        q.append((nr, nc))
            print(f"Tòa nhà #{k} tại ({br},{bc}):  [dist từ tòa này | dist_sum sau cộng | reach]")
            for r in range(m):
                print(f"  {row_str(r, dist)} | {row_str(r, dist_sum)} | {row_str(r, reach)}")
    cand = [(dist_sum[r][c], (r, c)) for r in range(m) for c in range(n) if grid[r][c] == EMPTY and reach[r][c] == k]
    print("Ô hợp lệ tốt nhất:", min(cand) if cand else "không có -> -1")


trace_from_buildings(ex1)

Tòa nhà #1 tại (0,0):  [dist từ tòa này | dist_sum sau cộng | reach]
   B  1  #  5  B |  B  1  #  5  B |  B  1  #  1  B
   1  2  3  4  5 |  1  2  3  4  5 |  1  1  1  1  1
   2  3  B  5  6 |  2  3  B  5  6 |  1  1  B  1  1
Tòa nhà #2 tại (0,4):  [dist từ tòa này | dist_sum sau cộng | reach]
   B  5  #  1  B |  B  6  #  6  B |  B  2  #  2  B
   5  4  3  2  1 |  6  6  6  6  6 |  2  2  2  2  2
   6  5  B  3  2 |  8  8  B  8  8 |  2  2  B  2  2
Tòa nhà #3 tại (2,2):  [dist từ tòa này | dist_sum sau cộng | reach]
   B  3  #  3  B |  B  9  #  9  B |  B  3  #  3  B
   3  2  1  2  3 |  9  8  7  8  9 |  3  3  3  3  3
   2  1  B  1  2 | 10  9  B  9 10 |  3  3  B  3  3
Ô hợp lệ tốt nhất: (7, (1, 2))


### 6.3 Trace trên lưới `pocket`: `reach` cứu ta khỏi đáp án sai

Lưới `pocket` có 3 tòa nhà `(0,1)`, `(2,4)`, `(3,0)`. Theo dõi ô `(0,0)` trong trace dưới:

- Sau tòa nhà `(0,1)`: `dist_sum[(0,0)] = 1`, `reach = 1`.
- Hai tòa nhà sau **không** tới được nó (cột `dist` hiện `.`), nên nó vẫn là `1`, `reach = 1`.

Kết quả cuối: ô tốt nhất hợp lệ là `(2,1)` với tổng `9 = 2 + 5 + 2` (2 bước tới `(0,1)`, 5 bước tới `(2,4)`, 2 bước tới `(3,0)` — bạn hãy tự tìm các đường đó trên hình; lưu ý tới `(2,4)` phải vòng xuống hàng 3 vì cột 3 có vật cản). Nếu quên `reach`, code sẽ trả `1` — sai.

In [16]:
show_grid(pocket, "pocket:")
trace_from_buildings(pocket)

pocket:
    c0 c1 c2 c3 c4
r0:  .  B  .  .  .
r1:  #  .  .  #  .
r2:  .  .  .  #  B
r3:  B  #  .  .  .
Tòa nhà #1 tại (0,1):  [dist từ tòa này | dist_sum sau cộng | reach]
   1  B  1  2  3 |  1  B  1  2  3 |  1  B  1  1  1
   #  1  2  #  4 |  #  1  2  #  4 |  #  1  1  #  1
   3  2  3  #  B |  3  2  3  #  B |  1  1  1  #  B
   B  #  4  5  6 |  B  #  4  5  6 |  B  #  1  1  1
Tòa nhà #2 tại (2,4):  [dist từ tòa này | dist_sum sau cộng | reach]
   .  B  4  3  2 |  1  B  5  5  5 |  1  B  2  2  2
   #  6  5  #  1 |  #  7  7  #  5 |  #  2  2  #  2
   6  5  4  #  B |  9  7  7  #  B |  2  2  2  #  B
   B  #  3  2  1 |  B  #  7  7  7 |  B  #  2  2  2
Tòa nhà #3 tại (3,0):  [dist từ tòa này | dist_sum sau cộng | reach]
   .  B  5  6  7 |  1  B 10 11 12 |  1  B  3  3  3
   #  3  4  #  8 |  # 10 11  # 13 |  #  3  3  #  3
   1  2  3  #  B | 10  9 10  #  B |  3  3  3  #  B
   B  #  4  5  6 |  B  # 11 12 13 |  B  #  3  3  3
Ô hợp lệ tốt nhất: (9, (2, 1))


### 6.4 Khi nào BFS từ **tòa nhà**, khi nào BFS từ **ô trống**?

Cả hai hướng đều đúng (nhờ tính đối xứng). Khác nhau ở **số lần chạy BFS**:

| Hướng | Số lần BFS | Mỗi lần BFS tốn | Tổng |
|---|:---:|---|---|
| Từ mỗi ô trống (Cách 1) | `E` | đi trên ô trống: `O(E)`, cộng cấp phát bảng `dist`: `O(m·n)` | `O(E · m·n)` |
| Từ mỗi tòa nhà (Cách 2) | `B` | như trên | `O(B · m·n)` |

Quy tắc: **chạy BFS từ phía có ít điểm hơn** → `O(min(B, E) · m · n)`.

- Lưới "thoáng" (ít tòa nhà, nhiều đất trống) — trường hợp thường gặp: BFS từ tòa nhà nhanh hơn hẳn. Ví dụ `30 × 30` với 4 tòa nhà ở 4 góc: 4 BFS thay vì 896 BFS.
- Lưới "đặc" (rất nhiều tòa nhà, ít đất trống): BFS từ ô trống lại lợi hơn.

Ý tưởng "chọn phía ít hơn để làm nguồn" xuất hiện ở nhiều bài khác (ví dụ BFS hai chiều trong *Word Ladder* luôn mở rộng phía nhỏ hơn).

Hàm `sd_auto` dưới đếm `B` và `E` rồi chọn hướng. Ta đo thời gian trên 2 lưới `30 × 30`: lưới thoáng `open_grid` và lưới đặc `dense_grid` (hàng tòa nhà – hàng trống – hàng tòa nhà lặp lại, thêm cột cuối trống để nối các hàng trống).

In [17]:
def sd_auto(grid):
    B = sum(row.count(BUILDING) for row in grid)
    E = sum(row.count(EMPTY) for row in grid)
    return sd_from_buildings(grid) if B <= E else sd_bruteforce(grid)


S = 30
open_grid = [[0] * S for _ in range(S)]
for r, c in [(0, 0), (0, S - 1), (S - 1, 0), (S - 1, S - 1)]:
    open_grid[r][c] = BUILDING
dense_grid = [[BUILDING if r % 3 != 1 else EMPTY for _ in range(S)] for r in range(S)]
for r in range(S):
    dense_grid[r][S - 1] = EMPTY

for name, g in [("open_grid ", open_grid), ("dense_grid", dense_grid)]:
    B = sum(row.count(BUILDING) for row in g)
    E = sum(row.count(EMPTY) for row in g)
    line = [f"{name}: B={B:>3}, E={E:>3}"]
    for label, fn in [("từ ô trống", sd_bruteforce), ("từ tòa nhà", sd_from_buildings), ("auto", sd_auto)]:
        t0 = time.perf_counter()
        ans = fn(g)
        line.append(f"{label}={ans} ({(time.perf_counter() - t0) * 1000:.0f} ms)")
    print(" | ".join(line))

open_grid : B=  4, E=896 | từ ô trống=116 (635 ms) | từ tòa nhà=116 (2 ms) | auto=116 (3 ms)


dense_grid: B=580, E=320 | từ ô trống=13610 (144 ms) | từ tòa nhà=13610 (163 ms) | auto=13610 (162 ms)


Con số mili-giây cụ thể tùy máy, nhưng xu hướng rõ: ở `open_grid`, BFS từ tòa nhà nhanh hơn **hàng trăm lần**; ở `dense_grid` (`B > E`), BFS từ ô trống lại nhỉnh hơn. Ba cách cho **cùng đáp án** ở cả hai lưới.

Trên LeetCode, đa số test là lưới thoáng, nên lời giải chuẩn chọn **BFS từ tòa nhà** (Cách 2). Phần tiếp theo sẽ tối ưu chính hướng này.

## 7. Code nộp LeetCode (Cách 2)

Chuyển `sd_from_buildings` sang đúng format `class Solution`. Một thay đổi nhỏ: thay vì bảng `dist` lưu số bước, ta dùng bảng `seen` kiểu `bool` và **mang số bước `d` theo trong hàng đợi** (`(r, c, d)`) — cách viết rất phổ biến, đỡ phải đọc lại `dist[r][c]`.

Signature giữ nguyên LeetCode: `def shortestDistance(self, grid: List[List[int]]) -> int`.

In [18]:
class Solution:
    def shortestDistance(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        dist_sum = [[0] * n for _ in range(m)]
        reach = [[0] * n for _ in range(m)]
        buildings = 0
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))

        for br in range(m):
            for bc in range(n):
                if grid[br][bc] != 1:
                    continue
                buildings += 1
                seen = [[False] * n for _ in range(m)]
                seen[br][bc] = True
                q = deque([(br, bc, 0)])
                while q:
                    r, c, d = q.popleft()
                    for dr, dc in dirs:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc] and grid[nr][nc] == 0:
                            seen[nr][nc] = True
                            dist_sum[nr][nc] += d + 1
                            reach[nr][nc] += 1
                            q.append((nr, nc, d + 1))

        return min((dist_sum[r][c] for r in range(m) for c in range(n)
                    if grid[r][c] == 0 and reach[r][c] == buildings), default=-1)

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<38} PASS ({actual})")


sol = Solution()
# Ví dụ của đề
check("Example 1", sol.shortestDistance(ex1), 7)
check("Example 2", sol.shortestDistance(ex2), 1)
check("Example 3", sol.shortestDistance(ex3), -1)
# Phản ví dụ của Manhattan (mục 3.2)
check("bức tường (Manhattan nói 4)", sol.shortestDistance(wall), 8)
check("tòa nhà chắn đường", sol.shortestDistance([[1, 0, 1, 0, 1]]), -1)
check("ô trống bị nhốt", sol.shortestDistance([[1, 1, 0]]), -1)
# Edge case
check("pocket (cần reach)", sol.shortestDistance(pocket), 9)
check("cột đơn [[1],[0]]", sol.shortestDistance([[1], [0]]), 1)
check("hai tòa nhà kề nhau, không ô trống", sol.shortestDistance([[1, 1]]), -1)
check("tòa nhà bị vật cản bao kín", sol.shortestDistance([[1, 2], [2, 0]]), -1)
check("nhà ở giữa 4 tòa nhà", sol.shortestDistance([[0, 1, 0], [1, 0, 1], [0, 1, 0]]), 4)
check("input không bị sửa", ex1 == [[1, 0, 2, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]], True)

Example 1                              PASS (7)
Example 2                              PASS (1)
Example 3                              PASS (-1)
bức tường (Manhattan nói 4)            PASS (8)
tòa nhà chắn đường                     PASS (-1)
ô trống bị nhốt                        PASS (-1)
pocket (cần reach)                     PASS (9)
cột đơn [[1],[0]]                      PASS (1)
hai tòa nhà kề nhau, không ô trống     PASS (-1)
tòa nhà bị vật cản bao kín             PASS (-1)
nhà ở giữa 4 tòa nhà                   PASS (4)
input không bị sửa                     PASS (True)


## 8. Cách 3 — Dùng chính `grid` làm dấu: chỉ đi vào ô mà **mọi tòa nhà trước** đã tới

### 8.1 Cách 2 còn lãng phí ở đâu?

Trong lưới `pocket`, ô `(0,0)` đã bị tòa nhà thứ 2 bỏ lỡ → nó **chắc chắn bị loại** (`reach` sẽ không bao giờ đủ). Vậy mà Cách 2 vẫn cho tòa nhà thứ 3, thứ 4, ... thử đi vào nó. Tổng quát hơn: một ô mà **một tòa nhà nào đó trước** không tới được thì **vô dụng** — không cần ai đi vào nữa.

Ngoài ra Cách 2 cấp phát một bảng `dist`/`seen` mới `m × n` cho **mỗi** tòa nhà, và một bảng `reach` riêng.

### 8.2 Ý tưởng: "giá trị bước được" giảm dần `0, -1, -2, ...`

Ta dùng **chính các ô trống của `grid`** để ghi lại "đã bao nhiêu tòa nhà tới được ô này", bằng **số âm**:

- Biến `walk` bắt đầu bằng `0`. Tòa nhà thứ nhất chỉ được bước lên ô có giá trị `== 0` (tức đất trống), và mỗi ô nó tới bị **giảm 1** → thành `-1`.
- Sau tòa nhà thứ nhất, `walk = -1`. Tòa nhà thứ hai chỉ được bước lên ô `== -1` — tức những ô **tòa nhà 1 đã tới** — và biến chúng thành `-2`.
- Tổng quát: tòa nhà thứ `k` chỉ bước lên ô `== -(k-1)`, biến chúng thành `-k`.

Vì sao dùng số **âm**? Để không trùng với `1` (tòa nhà) và `2` (vật cản): ô trống sau khi được đánh dấu vẫn **không bao giờ** bị nhầm thành tòa nhà hay vật cản.

Hai thứ "miễn phí" có được:

1. **Bỏ bảng `reach` và bảng `seen`**: giá trị `-k` chính là `reach = k`; và vì ô vừa đi qua đã đổi từ `walk` thành `walk - 1`, nó không còn thỏa `== walk` nữa → tự động không bị thăm lại trong lần BFS này.
2. **Dừng sớm**: trong lần BFS thứ `k`, ta tính luôn `best` = `dist_sum` nhỏ nhất trong các ô được thăm. Nếu **không thăm được ô nào** → không ô nào được cả `k` tòa nhà đầu tới → **trả `-1` ngay**, khỏi BFS các tòa nhà còn lại. Còn nếu đi hết, `best` của **lần BFS cuối** chính là đáp án (vì ô được thăm ở lần cuối là đúng những ô mọi tòa nhà đều tới).

Do ta **sửa `grid`**, trong notebook ta làm việc trên **bản sao** `g = [row[:] for row in grid]` để không phá input của người gọi (mục 13 sẽ cho thấy quên bản sao gây lỗi thế nào).

### 8.3 Vì sao đi trong "vùng bị thu hẹp" vẫn cho **đúng** khoảng cách?

Đây là câu hỏi quan trọng nhất của Cách 3. Tòa nhà `k` chỉ được đi trên các ô mà tòa nhà `1..k-1` đã tới. Lỡ như đường **ngắn nhất** từ tòa nhà `k` tới ô `X` phải đi qua một ô `Y` mà một tòa nhà trước **không** tới được thì sao — BFS bị cấm đi qua `Y` và sẽ đo sai?

Không thể xảy ra. Lý luận bằng **vùng liên thông** (connected component) của các ô trống:

- Gọi `C` là vùng các ô trống **nối được với `X`** chỉ qua ô trống (đi 4 hướng). Đường ngắn nhất từ tòa nhà `k` tới `X` có dạng `tòa nhà k → e1 → e2 → ... → X`, mọi ô `e1, e2, ...` đều là đất trống nối liền tới `X`, nên **đều nằm trong `C`**.
- Nếu một tòa nhà `j` tới được `X`, thì nó bước được vào một ô nào đó của `C`, và từ đó đi được tới **mọi** ô trong `C` (vì `C` liên thông). Vậy: *tòa nhà nào tới được `X` thì tới được cả vùng `C` của `X`.*
- Ta chỉ quan tâm `X` mà tòa nhà `1..k-1` **đều** tới được (các ô khác đằng nào cũng bị loại). Khi đó mọi ô của `C` cũng được `1..k-1` tới → mọi ô của `C` đang có giá trị `-(k-1)` → **đường ngắn nhất nằm trọn trong vùng được phép**. BFS thu hẹp đo **đúng**.

**Invariant**: *ngay trước lần BFS thứ `k`, một ô trống có giá trị `-(k-1)` khi và chỉ khi nó tới được từ cả `k-1` tòa nhà đầu; và `total` của nó bằng đúng tổng khoảng cách đi bộ tới `k-1` tòa nhà đó.*

In [19]:
def sd_walk(grid):
    g = [row[:] for row in grid]              # bản sao: không làm hỏng input
    m, n = len(g), len(g[0])
    total = [[0] * n for _ in range(m)]       # tổng quãng đường (thay cho dist_sum)
    walk = 0                                  # lần BFS này chỉ được bước lên ô == walk
    best = -1
    for br in range(m):
        for bc in range(n):
            if g[br][bc] != BUILDING:
                continue
            best = float("inf")
            q = deque([(br, bc, 0)])
            while q:
                r, c, d = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and g[nr][nc] == walk:
                        g[nr][nc] -= 1                 # thêm 1 tòa nhà tới được ô này
                        total[nr][nc] += d + 1
                        best = min(best, total[nr][nc])
                        q.append((nr, nc, d + 1))
            if best == float("inf"):                   # không ô nào được mọi tòa nhà tới
                return -1
            walk -= 1
    return best


for name, g in [("Example 1", ex1), ("Example 2", ex2), ("Example 3", ex3), ("wall", wall), ("pocket", pocket)]:
    print(f"{name:<10}: {sd_walk(g):>2} | Cách 2: {sd_from_buildings(g):>2}")

Example 1 :  7 | Cách 2:  7
Example 2 :  1 | Cách 2:  1
Example 3 : -1 | Cách 2: -1
wall      :  8 | Cách 2:  8
pocket    :  9 | Cách 2:  9


### 8.4 Trace Cách 3: lưới dấu sau từng tòa nhà

Trên `pocket`:

| Lần BFS | Tòa nhà | Chỉ bước lên ô `==` | Số ô thăm | Điều xảy ra |
|:---:|---|:---:|:---:|---|
| 1 | `(0,1)` | `0` | 13 | mọi ô trống đều tới được → tất cả thành `-1` |
| 2 | `(2,4)` | `-1` | 12 | túi `(0,0)` không tới được → **vẫn là `-1`**, các ô khác thành `-2` |
| 3 | `(3,0)` | `-2` | 12 | **không thèm** ghé `(0,0)` nữa; `best` của lần này = **9** |

Ví dụ thứ hai, lưới `enclosed` — tòa nhà `(2,2)` bị vật cản bao kín:

```text
       c0 c1 c2
r0:     B  .  .
r1:     .  .  #
r2:     .  #  B
```

Lần BFS 1 từ `(0,0)` thăm 5 ô. Lần BFS 2 từ `(2,2)`: không có hàng xóm nào `== -1` → thăm 0 ô → **trả `-1` ngay**.

Hàm `trace_walk` in lưới dấu (`B`, `#`, hoặc giá trị âm của ô trống) và bảng `total` sau mỗi lần BFS.

In [20]:
def trace_walk(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    total = [[0] * n for _ in range(m)]
    walk, k = 0, 0
    best = -1

    def row_str(r, table):
        return " ".join(f"{(SYMBOL[g[r][c]] if g[r][c] > 0 else str(table[r][c])):>2}" for c in range(n))

    for br in range(m):
        for bc in range(n):
            if g[br][bc] != BUILDING:
                continue
            k += 1
            best, visited = float("inf"), 0
            q = deque([(br, bc, 0)])
            while q:
                r, c, d = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and g[nr][nc] == walk:
                        g[nr][nc] -= 1
                        total[nr][nc] += d + 1
                        best = min(best, total[nr][nc])
                        visited += 1
                        q.append((nr, nc, d + 1))
            print(f"BFS #{k} từ ({br},{bc}), chỉ bước lên ô == {walk}: thăm {visited} ô, best = {best}")
            for r in range(m):
                print(f"    dấu: {row_str(r, g)}   | total: {row_str(r, total)}")
            if best == float("inf"):
                print("    -> không ô nào được mọi tòa nhà tới: trả -1 NGAY")
                return -1
            walk -= 1
    print("Đáp án =", best)
    return best


_ = trace_walk(pocket)

enclosed = [[1, 0, 0],
            [0, 0, 2],
            [0, 2, 1]]
_ = trace_walk(enclosed)

BFS #1 từ (0,1), chỉ bước lên ô == 0: thăm 13 ô, best = 1
    dấu: -1  B -1 -1 -1   | total:  1  B  1  2  3
    dấu:  # -1 -1  # -1   | total:  #  1  2  #  4
    dấu: -1 -1 -1  #  B   | total:  3  2  3  #  B
    dấu:  B  # -1 -1 -1   | total:  B  #  4  5  6
BFS #2 từ (2,4), chỉ bước lên ô == -1: thăm 12 ô, best = 5
    dấu: -1  B -2 -2 -2   | total:  1  B  5  5  5
    dấu:  # -2 -2  # -2   | total:  #  7  7  #  5
    dấu: -2 -2 -2  #  B   | total:  9  7  7  #  B
    dấu:  B  # -2 -2 -2   | total:  B  #  7  7  7
BFS #3 từ (3,0), chỉ bước lên ô == -2: thăm 12 ô, best = 9
    dấu: -1  B -3 -3 -3   | total:  1  B 10 11 12
    dấu:  # -3 -3  # -3   | total:  # 10 11  # 13
    dấu: -3 -3 -3  #  B   | total: 10  9 10  #  B
    dấu:  B  # -3 -3 -3   | total:  B  # 11 12 13
Đáp án = 9
BFS #1 từ (0,0), chỉ bước lên ô == 0: thăm 5 ô, best = 1
    dấu:  B -1 -1   | total:  B  1  2
    dấu: -1 -1  #   | total:  1  2  #
    dấu: -1  #  B   | total:  2  #  B
BFS #2 từ (2,2), chỉ bước lên ô == -1: thă

**Đọc trace:** ở `pocket`, sau BFS #2 ô `(0,0)` vẫn giữ dấu `-1` (không phải `-2`) — nó "rớt" khỏi cuộc đua và BFS #3 bỏ qua nó. Trên lưới nhỏ thì tiết kiệm ít, nhưng trên lưới lớn có nhiều vùng bị chia cắt, mỗi tòa nhà sau đi trên vùng **nhỏ dần**, và chỉ cần một tòa nhà bị cô lập là thuật toán dừng ngay.

Lưu ý: ở **trường hợp xấu nhất** (mọi tòa nhà tới được mọi ô) vùng không co lại, nên độ phức tạp vẫn là `O(B · m · n)` như Cách 2. Trên lưới "bình thường", Cách 3 chạy **ngang** Cách 2 (mục 10 sẽ đo); lợi thế thật của nó là **bớt bộ nhớ** (không cần `reach`, không cấp phát `seen` mới cho mỗi tòa nhà) và **dừng sớm** khi vùng ứng viên rỗng.

## 9. Cách 4 — Phát hiện sớm `-1`: tòa nhà phải "chạm" được mọi tòa nhà khác

### 9.1 Ý tưởng

Cách 3 dừng sớm khi một lần BFS **không thăm được ô nào**. Nhưng có trường hợp nó vẫn chạy rất lâu mới phát hiện: tưởng tượng 200 tòa nhà tới được nhau thoải mái, chỉ riêng **tòa nhà cuối cùng** (theo thứ tự quét) bị vật cản bao kín. Cách 2 và Cách 3 đều phải BFS 199 tòa nhà đầu rồi mới biết đáp án là `-1`.

**Quan sát:** nếu có một ô trống `X` tới được **mọi** tòa nhà, thì **hai tòa nhà bất kỳ** `P`, `Q` cũng "tới được nhau": đi `P → ... → X → ... → Q`, mọi ô ở giữa đều là đất trống. Lật ngược lại (phản đảo — contrapositive):

> Nếu BFS từ một tòa nhà `P` **không chạm được** hết `B - 1` tòa nhà còn lại, thì **không tồn tại** ô nhà hợp lệ → trả `-1` **ngay sau lần BFS đầu tiên** đó.

Nên chỉ cần, trong mỗi lần BFS, **đếm số tòa nhà khác bị chạm** (hàng xóm là `1`, đếm mỗi tòa nhà một lần). Đếm thiếu → `-1`.

Lưu ý đây là **điều kiện cần, chưa đủ**: các tòa nhà chạm được nhau vẫn chưa chắc có ô trống chung. Ví dụ `[[1, 1, 0]]`: hai tòa nhà kề nhau nên "chạm" được nhau trực tiếp, nhưng ô trống `(0,2)` chỉ tới được một tòa nhà → vẫn `-1`. Vì vậy ta **vẫn giữ** `reach` (hoặc dấu `walk`) như cũ.

Cài đặt trên nền Cách 2: tòa nhà chạm được cũng đánh dấu `dist` (để không đếm hai lần) nhưng **không** đưa vào hàng đợi. Tham số `log=True` in ra diễn biến để quan sát.

In [21]:
def sd_from_buildings_early(grid, log=False):
    m, n = len(grid), len(grid[0])
    total_buildings = sum(row.count(BUILDING) for row in grid)
    dist_sum = [[0] * n for _ in range(m)]
    reach = [[0] * n for _ in range(m)]
    bfs_count = 0
    for br in range(m):
        for bc in range(n):
            if grid[br][bc] != BUILDING:
                continue
            bfs_count += 1
            dist = [[-1] * n for _ in range(m)]
            dist[br][bc] = 0
            touched = 0                                   # số tòa nhà KHÁC chạm được
            q = deque([(br, bc)])
            while q:
                r, c = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < m and 0 <= nc < n) or dist[nr][nc] != -1:
                        continue
                    if grid[nr][nc] == EMPTY:
                        dist[nr][nc] = dist[r][c] + 1
                        dist_sum[nr][nc] += dist[nr][nc]
                        reach[nr][nc] += 1
                        q.append((nr, nc))
                    elif grid[nr][nc] == BUILDING:
                        dist[nr][nc] = dist[r][c] + 1     # đánh dấu để không đếm lại
                        touched += 1                      # chạm, nhưng KHÔNG đi xuyên
            if log:
                print(f"  BFS #{bfs_count} từ ({br},{bc}): chạm {touched}/{total_buildings - 1} tòa nhà khác")
            if touched < total_buildings - 1:
                if log:
                    print("  -> có tòa nhà không chạm được: trả -1 ngay")
                return -1
    return min((dist_sum[r][c] for r in range(m) for c in range(n)
                if grid[r][c] == EMPTY and reach[r][c] == total_buildings), default=-1)


print("enclosed:")
print("  kết quả:", sd_from_buildings_early(enclosed, log=True))
print("Example 1:")
print("  kết quả:", sd_from_buildings_early(ex1, log=True))
print("[[1, 1, 0]] (điều kiện cần chưa đủ):", sd_from_buildings_early([[1, 1, 0]]))

enclosed:
  BFS #1 từ (0,0): chạm 0/1 tòa nhà khác
  -> có tòa nhà không chạm được: trả -1 ngay
  kết quả: -1
Example 1:
  BFS #1 từ (0,0): chạm 2/2 tòa nhà khác
  BFS #2 từ (0,4): chạm 2/2 tòa nhà khác
  BFS #3 từ (2,2): chạm 2/2 tòa nhà khác
  kết quả: 7
[[1, 1, 0]] (điều kiện cần chưa đủ): -1


### 9.2 Ghép Cách 3 + Cách 4: bản tối ưu

Có thể đếm "tòa nhà chạm được" ngay trong BFS **thu hẹp** của Cách 3 không? Có, và lý luận vẫn là vùng liên thông ở mục 8.3: nếu tồn tại ô đáp án `X` với vùng `C`, thì ở lần BFS thứ `k` mọi ô của `C` đều đang mang dấu `-(k-1)`; tòa nhà `k` bước được vào `C`, đi khắp `C`, và **mọi** tòa nhà đều kề với `C` (vì đều tới được `X`) → BFS thu hẹp chạm đủ `B - 1` tòa nhà. Đếm thiếu → chắc chắn không có `X` → `-1`.

Khác biệt khi đếm trong bản dấu: tòa nhà vẫn mang giá trị `1` trên `g` (ta không sửa ô tòa nhà), nên cần một `set` nhỏ `seen_b` để không đếm một tòa nhà hai lần trong cùng một lần BFS, và bỏ qua chính tòa nhà nguồn.

Đo trên lưới `50 × 50` (đúng giới hạn đề) dựng sẵn: 200 tòa nhà rải ngẫu nhiên trên nền đất trống, và **góc dưới phải** là một tòa nhà bị vật cản bao kín (nó là tòa nhà **cuối cùng** theo thứ tự quét).

In [22]:
def sd_walk_early(grid):
    g = [row[:] for row in grid]
    m, n = len(g), len(g[0])
    total_buildings = sum(row.count(BUILDING) for row in g)
    total = [[0] * n for _ in range(m)]
    walk, best = 0, -1
    for br in range(m):
        for bc in range(n):
            if g[br][bc] != BUILDING:
                continue
            best = float("inf")
            seen_b = {(br, bc)}                       # tòa nhà đã chạm trong lần BFS này
            q = deque([(br, bc, 0)])
            while q:
                r, c, d = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if not (0 <= nr < m and 0 <= nc < n):
                        continue
                    if g[nr][nc] == walk:
                        g[nr][nc] -= 1
                        total[nr][nc] += d + 1
                        best = min(best, total[nr][nc])
                        q.append((nr, nc, d + 1))
                    elif g[nr][nc] == BUILDING:
                        seen_b.add((nr, nc))
            if best == float("inf") or len(seen_b) < total_buildings:   # seen_b gồm cả nguồn
                return -1
            walk -= 1
    return best


random.seed(317)
trap = [[0] * 50 for _ in range(50)]
for r, c in random.sample([(r, c) for r in range(47) for c in range(47)], 200):
    trap[r][c] = BUILDING
trap[49][49] = BUILDING
trap[48][49] = trap[49][48] = OBSTACLE           # nhốt tòa nhà cuối cùng

for name, fn in [("Cách 2", sd_from_buildings), ("Cách 3 (dấu)", sd_walk),
                 ("Cách 4 (Cách 2 + sớm)", sd_from_buildings_early), ("Cách 3 + 4", sd_walk_early)]:
    t0 = time.perf_counter()
    ans = fn(trap)
    print(f"{name:<22}: {ans:>3}  ({(time.perf_counter() - t0) * 1000:7.1f} ms)")

Cách 2                :  -1  (  369.3 ms)


Cách 3 (dấu)          :  -1  (  294.7 ms)
Cách 4 (Cách 2 + sớm) :  -1  (    1.5 ms)
Cách 3 + 4            :  -1  (    1.7 ms)


Cách 2 và Cách 3 phải BFS gần hết 201 tòa nhà mới biết là `-1`; hai bản có "phát hiện sớm" dừng ngay sau **một** lần BFS — nhanh hơn hàng trăm lần trên đúng loại test này.

### 9.3 Bản LeetCode tối ưu (Cách 3 + Cách 4)

Trên LeetCode được phép sửa `grid` trực tiếp (tiết kiệm bộ nhớ), nhưng cẩn thận: nếu ở môi trường thật input còn được dùng lại thì nên chép ra bản sao như trong notebook. Dưới đây giữ bản sao cho an toàn khi chạy nhiều test liên tiếp.

In [23]:
class SolutionFast:
    def shortestDistance(self, grid: List[List[int]]) -> int:
        g = [row[:] for row in grid]
        m, n = len(g), len(g[0])
        buildings = sum(row.count(1) for row in g)
        total = [[0] * n for _ in range(m)]
        dirs = ((-1, 0), (1, 0), (0, -1), (0, 1))
        walk, best = 0, -1
        for br in range(m):
            for bc in range(n):
                if g[br][bc] != 1:
                    continue
                best = float("inf")
                seen_b = {(br, bc)}
                q = deque([(br, bc, 0)])
                while q:
                    r, c, d = q.popleft()
                    for dr, dc in dirs:
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < m and 0 <= nc < n:
                            if g[nr][nc] == walk:
                                g[nr][nc] -= 1
                                total[nr][nc] += d + 1
                                best = min(best, total[nr][nc])
                                q.append((nr, nc, d + 1))
                            elif g[nr][nc] == 1:
                                seen_b.add((nr, nc))
                if best == float("inf") or len(seen_b) < buildings:
                    return -1
                walk -= 1
        return best


fast = SolutionFast()
for name, g, exp in [("Example 1", ex1, 7), ("Example 2", ex2, 1), ("Example 3", ex3, -1),
                     ("wall", wall, 8), ("pocket", pocket, 9), ("enclosed", enclosed, -1),
                     ("[[1,0,1,0,1]]", [[1, 0, 1, 0, 1]], -1), ("[[1,1,0]]", [[1, 1, 0]], -1),
                     ("trap 50x50", trap, -1)]:
    check(name, fast.shortestDistance(g), exp)

Example 1                              PASS (7)
Example 2                              PASS (1)
Example 3                              PASS (-1)
wall                                   PASS (8)
pocket                                 PASS (9)
enclosed                               PASS (-1)
[[1,0,1,0,1]]                          PASS (-1)
[[1,1,0]]                              PASS (-1)
trap 50x50                             PASS (-1)


## 10. Đối chiếu chéo các cách trên lưới ngẫu nhiên

Cách 1 (BFS từ mỗi ô trống) chậm nhưng làm **đúng nguyên văn đề**, nên là "trọng tài". Sinh 600 lưới ngẫu nhiên kích thước `1..5 × 1..5`, mỗi ô là `0` (đất trống, xác suất cao), `1` hoặc `2`; bảo đảm có **ít nhất một** tòa nhà (đúng constraints). `random.seed(317)` cố định để lần chạy nào cũng ra cùng bộ test.

Kiểm tra thêm: sau khi chạy tất cả các hàm, **lưới input không bị thay đổi** (bắt lỗi quên bản sao ở Cách 3).

Sau đó đo thời gian các cách nhanh trên vài lưới `50 × 50`. Con số tùy máy; điều đáng nhìn là: trên lưới thoáng hay lưới "vừa", ba bản chạy **ngang nhau** (cùng `O(B · N)`); chỉ ở lưới `trap` bản có **phát hiện sớm** mới bỏ xa hẳn.

In [24]:
random.seed(317)
impls = {
    "Cách 2": sd_from_buildings, "auto": sd_auto, "Cách 3": sd_walk,
    "Cách 4": sd_from_buildings_early, "Cách 3+4": sd_walk_early,
    "Solution": Solution().shortestDistance, "SolutionFast": SolutionFast().shortestDistance,
}
stats = {"-1": 0, "có đáp án": 0}
for _ in range(600):
    m, n = random.randint(1, 5), random.randint(1, 5)
    g = [[random.choice([0, 0, 0, 0, 1, 1, 2]) for _ in range(n)] for _ in range(m)]
    if not any(BUILDING in row for row in g):
        g[random.randrange(m)][random.randrange(n)] = BUILDING
    snapshot = [row[:] for row in g]
    expected = sd_bruteforce(g)
    for name, fn in impls.items():
        got = fn(g)
        assert got == expected, (name, g, got, expected)
    assert g == snapshot, "input bị sửa!"
    stats["-1" if expected == -1 else "có đáp án"] += 1
print("600 lưới ngẫu nhiên: 7 cài đặt đều khớp brute force", stats)

600 lưới ngẫu nhiên: 7 cài đặt đều khớp brute force {'-1': 247, 'có đáp án': 353}


In [25]:
random.seed(2026)


def random_grid(m, n, p_building, p_obstacle):
    return [[BUILDING if (x := random.random()) < p_building else OBSTACLE if x < p_building + p_obstacle else EMPTY
             for _ in range(n)] for _ in range(m)]


big = {
    "thoáng: 10 tòa nhà, không vật cản": random_grid(50, 50, 0.004, 0.0),
    "vừa: 5% tòa nhà, 10% vật cản     ": random_grid(50, 50, 0.05, 0.10),
    "trap (tòa nhà cuối bị nhốt)      ": trap,
}
for label, g in big.items():
    line = []
    for name, fn in [("C2", sd_from_buildings), ("C3", sd_walk), ("C3+4", sd_walk_early)]:
        t0 = time.perf_counter()
        ans = fn(g)
        line.append(f"{name}={ans} ({(time.perf_counter() - t0) * 1000:5.0f} ms)")
    print(label, "|", " | ".join(line))

thoáng: 10 tòa nhà, không vật cản | C2=144 (   14 ms) | C3=144 (   11 ms) | C3+4=144 (   13 ms)


vừa: 5% tòa nhà, 10% vật cản      | C2=3174 (  210 ms) | C3=3174 (  161 ms) | C3+4=3174 (  187 ms)


trap (tòa nhà cuối bị nhốt)       | C2=-1 (  318 ms) | C3=-1 (  318 ms) | C3+4=-1 (    2 ms)


## 11. Vì sao các cách đúng?

Gọi `dist(X, P)` = số bước đi bộ ít nhất từ ô trống `X` tới tòa nhà `P` (vô cùng nếu không tới được). Đáp án = `min` trên các ô trống `X` tới được mọi tòa nhà của `S(X) = Σ_P dist(X, P)`.

**BFS đo đúng `dist`.** Hàng đợi luôn có dạng `k ... k, k+1 ... k+1` (mục 2.5), nên lần đầu một ô được phát hiện là qua lớp nhỏ nhất → `dist` gán lúc đó là nhỏ nhất. Ta chỉ đẩy vào hàng đợi các ô **đất trống**, nên các đường BFS tìm được đều hợp lệ (không xuyên tòa nhà / vật cản); tòa nhà chỉ được "chạm" ở bước cuối.

**Cách 1.** Làm đúng định nghĩa: với mỗi ô trống, BFS cho `dist(X, P)` với mọi tòa nhà `P` chạm được; `len(touched) == B` đúng nghĩa "tới được mọi tòa nhà".

**Cách 2.** Lưới là đồ thị **vô hướng** nên `dist(X, P) = dist(P, X)` (đi ngược đường). BFS từ `P` cộng đúng `dist(P, X)` vào `dist_sum[X]` cho **mọi** `X` mà `P` tới được, và tăng `reach[X]`. Sau `B` lần: `reach[X] == B` ⇔ `X` tới được mọi tòa nhà, và khi đó `dist_sum[X] = S(X)`. Lấy `min` trên đúng tập đó → đáp án; tập rỗng → `-1`.

**Cách 3.** Invariant (mục 8.3): *trước lần BFS thứ `k`, ô trống mang dấu `-(k-1)` ⇔ nó tới được từ cả `k-1` tòa nhà đầu, và `total` của nó = tổng khoảng cách tới chúng.* Duy trì được vì: (i) tòa nhà `k` chỉ bước lên ô dấu `-(k-1)` và đổi chúng thành `-k`; (ii) nhờ lý luận vùng liên thông, mọi ô "còn trong cuộc đua" mà tòa nhà `k` tới được thì nó tới được **bằng đường ngắn nhất nằm trọn trong vùng được phép**, nên khoảng cách cộng vào là đúng. Lần BFS cuối thăm đúng tập ô hợp lệ, `best` của nó là đáp án; lần nào thăm 0 ô thì tập hợp lệ đã rỗng → `-1`.

**Cách 4.** Nếu có ô hợp lệ `X` thì mọi cặp tòa nhà nối được qua `X` → BFS từ bất kỳ tòa nhà nào cũng chạm đủ `B - 1` tòa nhà khác (kể cả trong vùng thu hẹp của Cách 3, mục 9.2). Phản đảo: chạm thiếu → không có `X` → `-1` là đúng. Khi không chạm thiếu, thuật toán chạy tiếp y như Cách 2/3, nên kết quả không đổi.

## 12. Độ phức tạp và so sánh các cách

`N = m · n ≤ 2500`, `B` = số tòa nhà, `E` = số ô trống. Mỗi ô có tối đa 4 hàng xóm.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| Manhattan | cộng `|Δr|+|Δc|` | `O(E · B)` | `O(B)` | **SAI** khi có vật cản / tòa nhà chắn đường |
| 1. BFS từ mỗi ô trống | đúng nguyên văn đề | `O(E · N)` ≤ `O(N²)` | `O(N)` | trọng tài khi test |
| 2. **BFS từ mỗi tòa nhà** | đối xứng + `dist_sum`, `reach` | `O(B · N)` | `O(N)` | **cách chính**, dễ viết, dễ giải thích |
| auto | BFS từ phía ít hơn | `O(min(B, E) · N)` | `O(N)` | tốt khi lưới rất "đặc" tòa nhà |
| 3. Dấu `walk` trên `grid` | chỉ đi vào ô còn hy vọng | `O(B · N)` tệ nhất; nhanh hơn khi vùng bị chia cắt | `O(N)` (bỏ `reach`, `seen`) | dừng sớm khi vùng rỗng |
| 3 + 4. + phát hiện sớm | đếm tòa nhà chạm được | `O(B · N)` tệ nhất; `O(N)` nếu phát hiện `-1` ngay lần đầu | `O(N)` | bản LeetCode tối ưu |

Vì sao Cách 2 là `O(B · N)`: mỗi lần BFS, mỗi ô trống vào hàng đợi nhiều nhất một lần và xét 4 hàng xóm → `O(N)`; thêm cấp phát bảng `seen` `O(N)`. Có `B` lần BFS. Bước chọn `min` cuối là `O(N)`. Bộ nhớ: `dist_sum`, `reach`, `seen`, hàng đợi — mỗi thứ `O(N)`.

Với giới hạn `50 × 50`: trường hợp xấu nhất `B · N` cỡ `1250 · 2500 ≈ 3 triệu` lượt — chạy được trong Python. (Thật ra khi `B` lớn thì `E` nhỏ đi, nên mỗi BFS cũng ngắn lại.)

## 13. Lỗi thường gặp

### Lỗi 1: Cho phép đi **xuyên tòa nhà**

Viết điều kiện đi tiếp là `grid[nr][nc] != 2` (chỉ cấm vật cản) — tức coi tòa nhà như đường đi:

```python
if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != 2 and dist[nr][nc] == -1:   # SAI
```

Hậu quả: khoảng cách bị **ngắn** đi, và các ô "không tới được" bị coi là tới được. Với `[[1, 0, 1, 0, 1]]` đáp án đúng là `-1`, bản sai trả một số hữu hạn.

**Sửa:** chỉ đi tiếp vào ô `== 0`. Tòa nhà chỉ được **chạm** (nếu cần đếm), không đưa vào hàng đợi.

In [26]:
def sd_wrong_pass_buildings(grid):
    m, n = len(grid), len(grid[0])
    dist_sum = [[0] * n for _ in range(m)]
    reach = [[0] * n for _ in range(m)]
    B = 0
    for br in range(m):
        for bc in range(n):
            if grid[br][bc] != BUILDING:
                continue
            B += 1
            dist = [[-1] * n for _ in range(m)]
            dist[br][bc] = 0
            q = deque([(br, bc)])
            while q:
                r, c = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] != OBSTACLE and dist[nr][nc] == -1:  # SAI
                        dist[nr][nc] = dist[r][c] + 1
                        dist_sum[nr][nc] += dist[nr][nc]
                        reach[nr][nc] += 1
                        q.append((nr, nc))
    return min((dist_sum[r][c] for r in range(m) for c in range(n)
                if grid[r][c] == EMPTY and reach[r][c] == B), default=-1)


for g in [[[1, 0, 1, 0, 1]], ex1, [[1, 1, 0]]]:
    print(f"{str(g):<48} sai: {sd_wrong_pass_buildings(g):>2} | đúng: {sd_from_buildings(g):>2}")

[[1, 0, 1, 0, 1]]                                sai:  5 | đúng: -1
[[1, 0, 2, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]] sai:  7 | đúng:  7
[[1, 1, 0]]                                      sai:  3 | đúng: -1


### Lỗi 2: Quên bảng `reach` (chỉ lấy `min dist_sum`)

Như mục 5.2: ô bị một số tòa nhà bỏ lỡ có `dist_sum` **thiếu số hạng** → nhỏ giả tạo. Trên `pocket`, bản quên `reach` trả `1` (ô túi `(0,0)`) thay vì `9`.

Lỗi "anh em" ở Cách 3: lấy `min total` trên **mọi** ô trống sau vòng lặp, thay vì `best` của **lần BFS cuối** — cũng chọn nhầm ô bị bỏ lỡ.

**Sửa:** chỉ xét ô có `reach == B` (Cách 2), hoặc dùng `best` của lần BFS cuối (Cách 3).

In [27]:
def sd_wrong_no_reach(grid):
    m, n = len(grid), len(grid[0])
    dist_sum = [[0] * n for _ in range(m)]
    for br in range(m):
        for bc in range(n):
            if grid[br][bc] == BUILDING:
                d, _ = bfs_distances(grid, br, bc)
                for r in range(m):
                    for c in range(n):
                        if grid[r][c] == EMPTY and d[r][c] != -1:
                            dist_sum[r][c] += d[r][c]
    return min((dist_sum[r][c] for r in range(m) for c in range(n) if grid[r][c] == EMPTY), default=-1)


print("pocket  -> sai:", sd_wrong_no_reach(pocket), "| đúng:", sd_from_buildings(pocket))
print("[[1,1,0]] -> sai:", sd_wrong_no_reach([[1, 1, 0]]), "| đúng:", sd_from_buildings([[1, 1, 0]]))

pocket  -> sai: 1 | đúng: 9
[[1,1,0]] -> sai: 1 | đúng: -1


### Lỗi 3: Đánh dấu "đã thăm" lúc **lấy ra** thay vì lúc **đưa vào**

```python
r, c = q.popleft()
if seen[r][c]: continue
seen[r][c] = True            # đánh dấu muộn
for ...: if grid[nr][nc] == 0 and not seen[nr][nc]: q.append((nr, nc))
```

Kết quả vẫn **đúng**, nhưng một ô có thể bị đẩy vào hàng đợi **nhiều lần** (mỗi hàng xóm đã thăm đẩy một lần) → hàng đợi phình to; ở ví dụ dưới số lần `append` gần **gấp đôi**. Còn nếu **quên hẳn** đánh dấu thì BFS chạy **vô hạn** (đẩy qua đẩy lại giữa hai ô kề nhau).

Cell dưới đếm số lần `append` của hai kiểu trên một lưới trống `20 × 20` có một tòa nhà ở góc.

**Sửa:** đánh dấu **ngay khi `append`**.

In [28]:
def count_pushes(grid, sr, sc, mark_on_push):
    m, n = len(grid), len(grid[0])
    seen = [[False] * n for _ in range(m)]
    q = deque([(sr, sc)])
    seen[sr][sc] = mark_on_push
    pushes = 1
    while q:
        r, c = q.popleft()
        if not mark_on_push:
            if seen[r][c] and (r, c) != (sr, sc):
                continue
            seen[r][c] = True
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == EMPTY and not seen[nr][nc]:
                if mark_on_push:
                    seen[nr][nc] = True
                q.append((nr, nc))
                pushes += 1
    return pushes


g20 = [[0] * 20 for _ in range(20)]
g20[0][0] = BUILDING
print("đánh dấu lúc append :", count_pushes(g20, 0, 0, True), "lần append (= số ô + 1)")
print("đánh dấu lúc popleft:", count_pushes(g20, 0, 0, False), "lần append")

đánh dấu lúc append : 400 lần append (= số ô + 1)
đánh dấu lúc popleft: 761 lần append


### Lỗi 4: Cách 3 sửa `grid` mà **không chép bản sao**

Bản "dấu `walk`" ghi số âm vào lưới. Nếu sửa thẳng lên `grid` của người gọi, lần gọi thứ hai trên **cùng** lưới sẽ thấy toàn số âm — không ô nào `== 0` — và trả `-1`. Trên LeetCode mỗi test có lưới riêng nên không lộ ra, nhưng trong notebook/test của bạn (và trong code thật) đây là lỗi khó tìm.

**Sửa:** `g = [row[:] for row in grid]` ở đầu hàm. Lưu ý `grid[:]` hay `list(grid)` **chưa đủ** — chỉ chép list ngoài, các hàng bên trong vẫn dùng chung.

In [29]:
def sd_walk_nocopy(grid):
    g = grid                                  # SAI: không chép
    m, n = len(g), len(g[0])
    total = [[0] * n for _ in range(m)]
    walk, best = 0, -1
    for br in range(m):
        for bc in range(n):
            if g[br][bc] != BUILDING:
                continue
            best = float("inf")
            q = deque([(br, bc, 0)])
            while q:
                r, c, d = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and g[nr][nc] == walk:
                        g[nr][nc] -= 1
                        total[nr][nc] += d + 1
                        best = min(best, total[nr][nc])
                        q.append((nr, nc, d + 1))
            if best == float("inf"):
                return -1
            walk -= 1
    return best


test_grid = [row[:] for row in ex1]
print("lần 1:", sd_walk_nocopy(test_grid))
print("lần 2:", sd_walk_nocopy(test_grid), " <- sai vì lưới đã bị ghi đè:")
print(test_grid)
shallow = ex1[:]
print("ex1[:] có chép hàng không?", shallow[0] is not ex1[0])

lần 1: 7
lần 2: -1  <- sai vì lưới đã bị ghi đè:
[[1, -3, 2, -3, 1], [-3, -3, -3, -3, -3], [-3, -3, 1, -3, -3]]
ex1[:] có chép hàng không? False


### Các lỗi khác cần nhớ

- **Lỗi 5 — Dùng Manhattan.** Pass Example 1 nhưng fail khi có vật cản/tòa nhà chắn đường (mục 3.2). Có ô cấm đi qua → phải BFS.
- **Lỗi 6 — Trả `float("inf")` hoặc `0` thay vì `-1`** khi không có ô hợp lệ. Dùng `min(..., default=-1)` hoặc kiểm tra `best == float("inf")`.
- **Lỗi 7 — Lệch 1 bước (off-by-one).** Khi mang `d` trong hàng đợi, ô hàng xóm phải nhận `d + 1`, không phải `d`. Kiểm tra nhanh bằng Example 2 `[[1, 0]]` → phải ra `1`.
- **Lỗi 8 — Dùng chung một bảng `seen` cho mọi tòa nhà.** Tòa nhà thứ hai sẽ thấy mọi ô "đã thăm" và không đi được đâu. Mỗi lần BFS phải có `seen` mới (hoặc dùng mẹo `walk` của Cách 3, vốn tự "làm mới" nhờ giá trị giảm dần).
- **Lỗi 9 — Dùng `list.pop(0)` làm hàng đợi.** Đúng nhưng `O(độ dài)` mỗi lần (mục 2.3) → có thể TLE. Dùng `deque.popleft()`.
- **Lỗi 10 — Quên kiểm tra biên dưới `0 <= nr`.** Chỉ số âm của Python không báo lỗi mà "quấn" sang mép đối diện (mục 2.2).
- **Lỗi 11 — Chọn cả ô tòa nhà/vật cản làm nơi xây nhà.** Khi lấy `min`, nhớ điều kiện `grid[r][c] == 0`.

## 14. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nhận ra bài "BFS trên lưới, nhiều nguồn"

- Đề cho **lưới**, đi **4 hướng**, mỗi bước tốn như nhau, và có **ô cấm đi qua** → khoảng cách = **BFS** (không phải Manhattan).
- Cần khoảng cách giữa **một tập điểm A** (tòa nhà) và **mọi ô** → BFS **từ mỗi điểm của A** (phía ít hơn), cộng dồn kết quả vào bảng chung.
- Nếu chỉ cần khoảng cách tới **điểm gần nhất** của A (không phải tổng) → **multi-source BFS**: đưa **tất cả** điểm A vào hàng đợi cùng lúc, BFS **một lần** (`O(N)`).
- Đề hỏi ô "tới được **mọi** nguồn" → luôn kèm bộ đếm `reach` (hoặc mẹo dấu giảm dần).
- Lưới không có ô cấm, đi xuyên tự do → khoảng cách Manhattan, tách hàng/cột, **trung vị** (#296).

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 296. Best Meeting Point (Hard, Premium) | không có vật cản, hẹn ở bất kỳ ô nào → Manhattan + trung vị — Bài tập 5 |
| 286. Walls and Gates (Medium, Premium) | multi-source BFS từ mọi "cổng": khoảng cách tới cổng **gần nhất** |
| 542. 01 Matrix (Medium) | multi-source BFS: khoảng cách tới số 0 gần nhất |
| 1162. As Far from Land as Possible (Medium) | multi-source BFS từ mọi ô đất |
| 994. Rotting Oranges (Medium) | multi-source BFS theo lớp = theo phút |
| 1293. Shortest Path in a Grid with Obstacles Elimination (Hard) | BFS với trạng thái mở rộng `(r, c, số lần phá tường còn lại)` |
| 2061 / 2812 và các bài "BFS + điều kiện an toàn" | BFS lưới làm bước tiền xử lý rồi tối ưu trên kết quả |

## 15. Bài tập vận dụng

Làm theo thứ tự, tự làm xong rồi mới xem **Gợi ý** (mục 16) và **Đáp án** (mục 17).

### Bài 1 — Đoán output (không chạy code)

1. `sd_from_buildings([[1, 0, 1]])` trả về bao nhiêu?
2. `sd_from_buildings([[1, 2, 0]])` trả về bao nhiêu?
3. `sd_from_buildings([[0, 1, 0], [1, 0, 1], [0, 1, 0]])` trả về bao nhiêu? Ô góc `(0,0)` có `reach` bằng mấy?
4. `sd_from_buildings([[1, 1]])` trả về bao nhiêu? Còn `sd_from_buildings_early([[1, 1]])` dừng ở bước nào?
5. Trên lưới `wall` (mục 3.2), `sd_manhattan_wrong` cho `4`. Ô nào nó "chọn"? Khoảng cách đi bộ thật từ ô đó tới hai tòa nhà là bao nhiêu?

### Bài 2 — Tính tay bằng bảng

Với lưới

```text
H2 = [[1, 0, 0, 0, 0],
      [0, 2, 2, 2, 0],
      [0, 0, 0, 1, 0]]
```

a) Vẽ lưới bằng ký hiệu `B`, `#`, `.`. Tính tay bảng khoảng cách đi bộ từ tòa nhà `(0,0)` và từ tòa nhà `(2,3)`.

b) Lập bảng `dist_sum`. Đáp án là bao nhiêu, đạt ở những ô nào?

c) Chạy tay Cách 3: sau lần BFS 1 và lần BFS 2, lưới dấu trông thế nào?

### Bài 3 — Điền TODO: BFS một nguồn trên lưới

Viết hàm `bfs_grid_exercise(grid, sr, sc)` trả về bảng `dist` (`-1` nếu không tới được), **chỉ đi qua ô `0`**, nguồn có `dist = 0`. Không được dùng `bfs_distances` có sẵn.

In [30]:
def bfs_grid_exercise(grid, sr, sc):
    # TODO 1: m, n; tạo dist toàn -1 bằng list comprehension (KHÔNG dùng [[-1]*n]*m)
    # TODO 2: dist[sr][sc] = 0; q = deque([(sr, sc)])
    # TODO 3: while q: lấy (r, c) bằng popleft
    # TODO 4: duyệt 4 hướng; nếu (nr, nc) trong lưới, grid == 0 và dist == -1:
    #         gán dist = dist[r][c] + 1 rồi append
    # TODO 5: return dist
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert bfs_grid_exercise(maze, 0, 0) == [[0, 1, 2, 3], [-1, -1, 3, -1], [6, 5, 4, 5]]
# assert bfs_grid_exercise([[1, 2], [2, 0]], 0, 0) == [[0, -1], [-1, -1]]
# assert bfs_grid_exercise([[0, 0, 0]], 0, 1) == [[1, 0, 1]]

### Bài 4 — Tự viết lại Cách 2

Không nhìn mục 6, tự viết `shortest_distance_exercise(grid)`: BFS từ mỗi tòa nhà, cộng dồn `dist_sum` và `reach`, trả `min` trên ô trống có `reach == B`, hoặc `-1`.

In [31]:
def shortest_distance_exercise(grid):
    # TODO: dist_sum, reach, đếm B; BFS từ mỗi tòa nhà (bảng seen riêng mỗi lần);
    #       cộng d + 1 vào dist_sum và +1 vào reach khi tới ô trống mới;
    #       cuối cùng min(..., default=-1) trên ô trống có reach == B
    pass


# Tự làm xong rồi mới mở test:
# assert shortest_distance_exercise(ex1) == 7
# assert shortest_distance_exercise(ex2) == 1
# assert shortest_distance_exercise(ex3) == -1
# assert shortest_distance_exercise(wall) == 8
# assert shortest_distance_exercise(pocket) == 9
# assert shortest_distance_exercise([[1, 1, 0]]) == -1

### Bài 5 — Biến thể: LeetCode 296. Best Meeting Point (Premium)

Lưới chỉ gồm `0` và `1` (`1` = nhà một người bạn), có ít nhất 2 người bạn. Mọi người được đi **xuyên mọi ô**, điểm hẹn là **bất kỳ ô nào** (kể cả ô có nhà). Khoảng cách là **Manhattan**. Trả về tổng khoảng cách nhỏ nhất.

Ví dụ của đề #296: `[[1,0,0,0,1],[0,0,0,0,0],[0,0,1,0,0]]` → `6` (hẹn ở `(0,2)`: `2 + 2 + 2`); `[[1,1]]` → `1`.

Yêu cầu: lời giải `O(m · n)` (hoặc `O(m·n + k log k)`), **không BFS**. Gợi ý: hàng và cột tách được.

In [32]:
def min_total_distance(grid):
    # TODO 1: rows = danh sách hàng của mọi số 1 (duyệt theo hàng -> tự nhiên đã sắp xếp)
    # TODO 2: cols = danh sách cột của mọi số 1 (duyệt theo cột, hoặc sort)
    # TODO 3: chọn hàng trung vị, cột trung vị; cộng |r - median_r| và |c - median_c|
    pass


# Tự làm xong rồi mới mở test:
# assert min_total_distance([[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]]) == 6
# assert min_total_distance([[1, 1]]) == 1
# assert min_total_distance([[1, 0, 1, 0, 1]]) == 4

### Bài 6 — Mini challenge: tòa nhà có **số người** khác nhau, trả về cả **vị trí**

Mỗi tòa nhà `P` có `people[P]` người, ai cũng phải đi từ nhà mới tới tòa nhà của mình... nói cách khác, ta muốn cực tiểu **tổng có trọng số** `Σ people[P] · dist(X, P)`. Trả về cặp `(tổng nhỏ nhất, (r, c))`; nếu nhiều ô cùng tổng thì chọn `(r, c)` nhỏ nhất theo thứ tự (hàng trước, cột sau); nếu không có ô hợp lệ trả `(-1, None)`.

Ví dụ: Example 1 với `people = {(0,0): 1, (0,4): 1, (2,2): 5}` — tòa nhà `(2,2)` đông người nên nhà mới bị "kéo" về phía nó.

In [33]:
def best_house(grid, people):
    # TODO: như Cách 2, nhưng cộng people[(br, bc)] * (d + 1) vào dist_sum;
    #       cuối cùng chọn min theo cặp (tổng, (r, c)) trên ô trống có reach == B
    pass


# Tự làm xong rồi mới mở test:
# assert best_house(ex1, {(0, 0): 1, (0, 4): 1, (2, 2): 1}) == (7, (1, 2))
# assert best_house(ex1, {(0, 0): 1, (0, 4): 1, (2, 2): 5}) == (11, (1, 2))
# assert best_house(ex1, {(0, 0): 5, (0, 4): 1, (2, 2): 1}) == (13, (0, 1))   # hòa với (1,0) -> chọn (0,1)
# assert best_house([[1, 1, 0]], {(0, 0): 1, (0, 1): 1}) == (-1, None)

## 16. Gợi ý theo mức

**Bài 1**
- Mức 1: với mỗi câu, vẽ lưới ra và tự hỏi: ô trống nào chạm được những tòa nhà nào?
- Mức 2: câu 3 — ô giữa `(1,1)` kề cả 4 tòa nhà; ô góc chỉ kề 2 tòa nhà và không đi tiếp được (hai hàng xóm đều là tòa nhà).
- Mức 3: câu 4 — không có ô trống thì `min` trên tập rỗng trả `default`. Với bản "sớm": hai tòa nhà kề nhau có chạm nhau không?

**Bài 2**
- Mức 1: bức tường `#` ở giữa hàng 1 buộc mọi đường phải đi vòng: hoặc vòng **trái–dưới**, hoặc vòng **trên–phải**.
- Mức 2: vòng trái–dưới gồm `(1,0),(2,0),(2,1),(2,2)`; vòng trên–phải gồm `(0,1),(0,2),(0,3),(0,4),(1,4),(2,4)`.
- Mức 3: đường nối hai tòa nhà đi vòng trên–phải dài bao nhiêu? vòng trái–dưới dài bao nhiêu? Ô nằm **trên** một đường ngắn nhất nối hai tòa nhà có tổng bằng độ dài đường đó.

**Bài 3**
- Mức 1: chép khung BFS ở mục 2.4, bỏ phần `touched`.
- Mức 2: `dist = [[-1] * n for _ in range(m)]`; điều kiện `0 <= nr < m and 0 <= nc < n and grid[nr][nc] == 0 and dist[nr][nc] == -1`.
- Mức 3: test thứ hai: nguồn là tòa nhà bị chắn hết → chỉ nguồn có `0`.

**Bài 4**
- Mức 1: hai vòng `for` ngoài tìm tòa nhà; bên trong là BFS của Bài 3, thêm 2 dòng cộng dồn.
- Mức 2: mang `d` trong hàng đợi: `q = deque([(br, bc, 0)])`, ô mới nhận `d + 1`.
- Mức 3: `return min((dist_sum[r][c] for r in range(m) for c in range(n) if grid[r][c] == 0 and reach[r][c] == B), default=-1)`.

**Bài 5**
- Mức 1: `|r1 - r2| + |c1 - c2|` → tổng = (tổng theo hàng) + (tổng theo cột), hai phần chọn **độc lập**.
- Mức 2: trên một trục số, điểm làm `Σ |x - xi|` nhỏ nhất là **trung vị** của các `xi`.
- Mức 3: duyệt hàng-rồi-cột thu `rows` (đã tăng dần); duyệt cột-rồi-hàng thu `cols` (đã tăng dần); `median = arr[len(arr) // 2]`.

**Bài 6**
- Mức 1: chỉ khác Cách 2 đúng một dòng: số cộng vào `dist_sum` nhân thêm `people[(br, bc)]`.
- Mức 2: so sánh tuple `(tổng, (r, c))` của Python tự xử lý luật "hòa thì chọn ô nhỏ hơn".
- Mức 3: `min(candidates, default=(-1, None))`.

## 17. Đáp án tham khảo

### Đáp án Bài 1

1. `2`: ô `(0,1)` cách mỗi tòa nhà 1 bước.
2. `-1`: tòa nhà `(0,0)` chỉ có hàng xóm `(0,1)` là vật cản → không tới được ô trống nào.
3. `4`: ô giữa `(1,1)` kề cả 4 tòa nhà, mỗi cái 1 bước. Ô góc `(0,0)` chỉ được tòa nhà `(0,1)` và `(1,0)` bước vào → `reach = 2 < 4`, bị loại.
4. `-1` (không có ô trống; `min` rỗng → `default=-1`). Bản "sớm": BFS từ `(0,0)` chạm được `(0,1)` (kề nhau) → đủ `1/1`; BFS từ `(0,1)` cũng chạm đủ → không dừng sớm; cuối cùng `min` rỗng → `-1`. Đây là ví dụ "điều kiện cần chưa đủ".
5. Manhattan cho `4` ở ô `(0,1)` (và `(2,1)`): `1 + 3`. Đi bộ thật từ `(0,1)`: `1` bước tới `(0,0)`, `7` bước tới `(2,0)` (vòng chữ U) → `8`.

In [34]:
print("1.", sd_from_buildings([[1, 0, 1]]))
print("2.", sd_from_buildings([[1, 2, 0]]))
g3 = [[0, 1, 0], [1, 0, 1], [0, 1, 0]]
print("3.", sd_from_buildings(g3), "| totals từng ô:")
totals_table(g3)
print("4.", sd_from_buildings([[1, 1]]))
print("   bản sớm:", sd_from_buildings_early([[1, 1]], log=True))
d_a, _ = bfs_distances(wall, 0, 0)
d_b, _ = bfs_distances(wall, 2, 0)
print("5. ô (0,1): Manhattan =", manhattan((0, 1), (0, 0)) + manhattan((0, 1), (2, 0)),
      "| đi bộ =", d_a[0][1], "+", d_b[0][1], "=", d_a[0][1] + d_b[0][1])

1. 2
2. -1
3. 4 | totals từng ô:
 --   B  --
  B   4   B
 --   B  --
4. -1
  BFS #1 từ (0,0): chạm 1/1 tòa nhà khác
  BFS #2 từ (0,1): chạm 1/1 tòa nhà khác
   bản sớm: -1
5. ô (0,1): Manhattan = 4 | đi bộ = 1 + 7 = 8


### Đáp án Bài 2

a) Lưới và hai bảng khoảng cách (`B`, `#` giữ nguyên):

```text
H2:              từ (0,0):          từ (2,3):
B . . . .        B 1 2 3 4          B 6 5 4 3
. # # # .        1 # # # 5          4 # # # 2
. . . B .        2 3 4 B 6          3 2 1 B 1
```

b) `dist_sum` (cộng hai bảng):

```text
B 7 7 7 7
5 # # # 7
5 5 5 B 7
```

Đáp án **5**, đạt ở 4 ô vòng trái–dưới `(1,0), (2,0), (2,1), (2,2)`. Lý do: đường ngắn nhất nối hai tòa nhà đi vòng trái–dưới dài **5** (`(0,0)→(1,0)→(2,0)→(2,1)→(2,2)→(2,3)`), còn vòng trên–phải dài **7**. Ô nằm trên một đường đi nối hai tòa nhà có tổng đúng bằng độ dài đường đó: vòng trái–dưới cho 5, vòng trên–phải cho 7. (Trong `trace_from_buildings` bên dưới, ô tốt nhất được in là `(1,0)` — ô nhỏ nhất theo thứ tự trong 4 ô hòa nhau.)

c) Lần BFS 1 (từ `(0,0)`, bước lên ô `== 0`): cả 10 ô trống thành `-1`. Lần BFS 2 (từ `(2,3)`, bước lên ô `== -1`): cả 10 ô thành `-2`, `best = 5`.

In [35]:
H2 = [[1, 0, 0, 0, 0],
      [0, 2, 2, 2, 0],
      [0, 0, 0, 1, 0]]
show_grid(H2, "H2:")
trace_from_buildings(H2)
_ = trace_walk(H2)

H2:
    c0 c1 c2 c3 c4
r0:  B  .  .  .  .
r1:  .  #  #  #  .
r2:  .  .  .  B  .
Tòa nhà #1 tại (0,0):  [dist từ tòa này | dist_sum sau cộng | reach]
   B  1  2  3  4 |  B  1  2  3  4 |  B  1  1  1  1
   1  #  #  #  5 |  1  #  #  #  5 |  1  #  #  #  1
   2  3  4  B  6 |  2  3  4  B  6 |  1  1  1  B  1
Tòa nhà #2 tại (2,3):  [dist từ tòa này | dist_sum sau cộng | reach]
   B  6  5  4  3 |  B  7  7  7  7 |  B  2  2  2  2
   4  #  #  #  2 |  5  #  #  #  7 |  2  #  #  #  2
   3  2  1  B  1 |  5  5  5  B  7 |  2  2  2  B  2
Ô hợp lệ tốt nhất: (5, (1, 0))
BFS #1 từ (0,0), chỉ bước lên ô == 0: thăm 10 ô, best = 1
    dấu:  B -1 -1 -1 -1   | total:  B  1  2  3  4
    dấu: -1  #  #  # -1   | total:  1  #  #  #  5
    dấu: -1 -1 -1  B -1   | total:  2  3  4  B  6
BFS #2 từ (2,3), chỉ bước lên ô == -1: thăm 10 ô, best = 5
    dấu:  B -2 -2 -2 -2   | total:  B  7  7  7  7
    dấu: -2  #  #  # -2   | total:  5  #  #  #  7
    dấu: -2 -2 -2  B -2   | total:  5  5  5  B  7
Đáp án = 5


### Đáp án Bài 3 và Bài 4

In [36]:
def bfs_grid_answer(grid, sr, sc):
    m, n = len(grid), len(grid[0])
    dist = [[-1] * n for _ in range(m)]
    dist[sr][sc] = 0
    q = deque([(sr, sc)])
    while q:
        r, c = q.popleft()
        for dr, dc in DIRS:
            nr, nc = r + dr, c + dc
            if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == EMPTY and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist


def shortest_distance_answer(grid):
    m, n = len(grid), len(grid[0])
    dist_sum = [[0] * n for _ in range(m)]
    reach = [[0] * n for _ in range(m)]
    B = 0
    for br in range(m):
        for bc in range(n):
            if grid[br][bc] != BUILDING:
                continue
            B += 1
            seen = [[False] * n for _ in range(m)]
            seen[br][bc] = True
            q = deque([(br, bc, 0)])
            while q:
                r, c, d = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == EMPTY and not seen[nr][nc]:
                        seen[nr][nc] = True
                        dist_sum[nr][nc] += d + 1
                        reach[nr][nc] += 1
                        q.append((nr, nc, d + 1))
    return min((dist_sum[r][c] for r in range(m) for c in range(n)
                if grid[r][c] == EMPTY and reach[r][c] == B), default=-1)


assert bfs_grid_answer(maze, 0, 0) == [[0, 1, 2, 3], [-1, -1, 3, -1], [6, 5, 4, 5]]
assert bfs_grid_answer([[1, 2], [2, 0]], 0, 0) == [[0, -1], [-1, -1]]
assert bfs_grid_answer([[0, 0, 0]], 0, 1) == [[1, 0, 1]]
assert shortest_distance_answer(ex1) == 7
assert shortest_distance_answer(ex2) == 1
assert shortest_distance_answer(ex3) == -1
assert shortest_distance_answer(wall) == 8
assert shortest_distance_answer(pocket) == 9
assert shortest_distance_answer([[1, 1, 0]]) == -1
print("Bài 3 và Bài 4: PASS")

Bài 3 và Bài 4: PASS


### Đáp án Bài 5 — Best Meeting Point (#296)

Tổng Manhattan tách thành **tổng theo hàng + tổng theo cột**, hai phần độc lập. Trên một trục số, `Σ |x - xi|` nhỏ nhất khi `x` là **trung vị** (median): nếu dịch `x` sang phải một đơn vị, các điểm bên trái `x` mỗi cái xa thêm 1, các điểm bên phải mỗi cái gần lại 1 — chỉ có lợi khi bên phải còn **nhiều** điểm hơn bên trái; trung vị là chỗ hai bên cân bằng.

Duyệt hàng-rồi-cột thu được `rows` tăng dần; duyệt cột-rồi-hàng thu được `cols` tăng dần → không cần sort, `O(m·n)`.

Đối chiếu với bài #317: ta kiểm tra trên các lưới **không vật cản và không có tòa nhà chắn đường** kiểu gì cũng được (#296 cho hẹn ở mọi ô, đi xuyên mọi ô) bằng brute force Manhattan trên mọi ô.

In [37]:
def min_total_distance_answer(grid):
    m, n = len(grid), len(grid[0])
    rows = [r for r in range(m) for c in range(n) if grid[r][c] == 1]   # đã tăng dần
    cols = [c for c in range(n) for r in range(m) if grid[r][c] == 1]   # đã tăng dần
    mr, mc = rows[len(rows) // 2], cols[len(cols) // 2]
    return sum(abs(r - mr) for r in rows) + sum(abs(c - mc) for c in cols)


def min_total_distance_brute(grid):
    m, n = len(grid), len(grid[0])
    homes = [(r, c) for r in range(m) for c in range(n) if grid[r][c] == 1]
    return min(sum(manhattan((r, c), h) for h in homes) for r in range(m) for c in range(n))


assert min_total_distance_answer([[1, 0, 0, 0, 1], [0, 0, 0, 0, 0], [0, 0, 1, 0, 0]]) == 6
assert min_total_distance_answer([[1, 1]]) == 1
assert min_total_distance_answer([[1, 0, 1, 0, 1]]) == 4
random.seed(296)
for _ in range(300):
    m, n = random.randint(1, 5), random.randint(1, 5)
    g = [[int(random.random() < 0.3) for _ in range(n)] for _ in range(m)]
    if sum(map(sum, g)) < 2:
        continue
    assert min_total_distance_answer(g) == min_total_distance_brute(g), g
print("Bài 5: PASS | #296 trên [[1,0,1,0,1]] =", min_total_distance_answer([[1, 0, 1, 0, 1]]),
      "nhưng #317 trên cùng lưới =", sd_from_buildings([[1, 0, 1, 0, 1]]))

Bài 5: PASS | #296 trên [[1,0,1,0,1]] = 4 nhưng #317 trên cùng lưới = -1


### Đáp án Bài 6 — Tổng có trọng số và vị trí

Chỉ khác Cách 2 ở dòng cộng dồn (nhân `w = people[(br, bc)]`) và ở bước chọn: so sánh tuple `(tổng, (r, c))` — Python so phần tử đầu trước, hòa mới so phần tử sau, đúng luật "hòa thì chọn ô nhỏ hơn".

Tính tay để kiểm tra (dùng các bảng khoảng cách ở mục 6.2):

- `people = {(0,0): 1, (0,4): 1, (2,2): 5}`: ô `(1,2)` có `3 + 3 + 5·1 = 11`; ô `(1,1)` có `2 + 4 + 5·2 = 16`; ô `(2,1)` có `3 + 5 + 5·1 = 13` → `(1,2)` thắng với **11**.
- `people = {(0,0): 5, (0,4): 1, (2,2): 1}`: nhà mới bị kéo về phía `(0,0)`. Ô `(0,1)` có `5·1 + 5 + 3 = 13`; ô `(1,0)` có `5·1 + 5 + 3 = 13` — **hòa**! Luật tie-break chọn `(0,1)` vì `(0,1) < (1,0)` (hàng 0 trước hàng 1). Đáp án `(13, (0, 1))`.

Cell dưới in kết quả cùng một brute force độc lập (BFS từ mỗi ô trống) để đối chiếu.

In [38]:
def best_house_answer(grid, people):
    m, n = len(grid), len(grid[0])
    dist_sum = [[0] * n for _ in range(m)]
    reach = [[0] * n for _ in range(m)]
    B = 0
    for br in range(m):
        for bc in range(n):
            if grid[br][bc] != BUILDING:
                continue
            B += 1
            w = people[(br, bc)]
            seen = [[False] * n for _ in range(m)]
            seen[br][bc] = True
            q = deque([(br, bc, 0)])
            while q:
                r, c, d = q.popleft()
                for dr, dc in DIRS:
                    nr, nc = r + dr, c + dc
                    if 0 <= nr < m and 0 <= nc < n and grid[nr][nc] == EMPTY and not seen[nr][nc]:
                        seen[nr][nc] = True
                        dist_sum[nr][nc] += w * (d + 1)
                        reach[nr][nc] += 1
                        q.append((nr, nc, d + 1))
    candidates = [(dist_sum[r][c], (r, c)) for r in range(m) for c in range(n)
                  if grid[r][c] == EMPTY and reach[r][c] == B]
    return min(candidates, default=(-1, None))


def best_house_brute(grid, people):
    m, n = len(grid), len(grid[0])
    best = (-1, None)
    for r in range(m):
        for c in range(n):
            if grid[r][c] == EMPTY:
                _, touched = bfs_distances(grid, r, c)
                if len(touched) == len(people):
                    cand = (sum(people[p] * d for p, d in touched.items()), (r, c))
                    best = cand if best[1] is None or cand < best else best
    return best


for ppl in [{(0, 0): 1, (0, 4): 1, (2, 2): 1}, {(0, 0): 1, (0, 4): 1, (2, 2): 5}, {(0, 0): 5, (0, 4): 1, (2, 2): 1}]:
    print(ppl, "->", best_house_answer(ex1, ppl), "| brute:", best_house_brute(ex1, ppl))
assert best_house_answer(ex1, {(0, 0): 1, (0, 4): 1, (2, 2): 1}) == (7, (1, 2))
assert best_house_answer(ex1, {(0, 0): 1, (0, 4): 1, (2, 2): 5}) == (11, (1, 2))
assert best_house_answer(ex1, {(0, 0): 5, (0, 4): 1, (2, 2): 1}) == (13, (0, 1))
assert best_house_answer([[1, 1, 0]], {(0, 0): 1, (0, 1): 1}) == (-1, None)
random.seed(6)
for _ in range(300):
    m, n = random.randint(1, 4), random.randint(1, 4)
    g = [[random.choice([0, 0, 0, 1, 2]) for _ in range(n)] for _ in range(m)]
    if not any(BUILDING in row for row in g):
        g[0][0] = BUILDING
    ppl = {(r, c): random.randint(1, 4) for r in range(m) for c in range(n) if g[r][c] == BUILDING}
    assert best_house_answer(g, ppl) == best_house_brute(g, ppl), (g, ppl)
print("Bài 6: PASS (khớp brute force trên 300 lưới ngẫu nhiên)")

{(0, 0): 1, (0, 4): 1, (2, 2): 1} -> (7, (1, 2)) | brute: (7, (1, 2))
{(0, 0): 1, (0, 4): 1, (2, 2): 5} -> (11, (1, 2)) | brute: (11, (1, 2))
{(0, 0): 5, (0, 4): 1, (2, 2): 1} -> (13, (0, 1)) | brute: (13, (0, 1))
Bài 6: PASS (khớp brute force trên 300 lưới ngẫu nhiên)


## 18. Template cuối cùng nên nhớ

```python
class Solution:
    def shortestDistance(self, grid: List[List[int]]) -> int:
        m, n = len(grid), len(grid[0])
        dist_sum = [[0] * n for _ in range(m)]
        reach = [[0] * n for _ in range(m)]
        B = 0
        for br in range(m):
            for bc in range(n):
                if grid[br][bc] != 1:
                    continue
                B += 1
                seen = [[False] * n for _ in range(m)]     # bảng MỚI cho mỗi tòa nhà
                seen[br][bc] = True
                q = deque([(br, bc, 0)])
                while q:
                    r, c, d = q.popleft()
                    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
                        nr, nc = r + dr, c + dc
                        if 0 <= nr < m and 0 <= nc < n and not seen[nr][nc] and grid[nr][nc] == 0:
                            seen[nr][nc] = True             # đánh dấu khi append
                            dist_sum[nr][nc] += d + 1
                            reach[nr][nc] += 1
                            q.append((nr, nc, d + 1))
        return min((dist_sum[r][c] for r in range(m) for c in range(n)
                    if grid[r][c] == 0 and reach[r][c] == B), default=-1)
```

Bản tối ưu (Cách 3 + 4): *thay `seen`/`reach` bằng dấu `walk = 0, -1, -2, ...` trên bản sao của `grid` — tòa nhà thứ `k` chỉ bước lên ô `== walk`, giảm ô đó đi 1; lần BFS nào không thăm được ô nào, hoặc không chạm đủ `B - 1` tòa nhà khác → trả `-1`; `best` của lần BFS cuối là đáp án.*

**Câu thần chú**: *"Có ô cấm thì BFS, không phải Manhattan. Đối xứng nên BFS từ tòa nhà (phía ít hơn). Cộng dồn `dist_sum`, đếm `reach`. Chỉ chọn ô mà mọi tòa nhà đều tới."*

## 19. Checklist tự kiểm tra trước khi submit

- Khoảng cách tính bằng **BFS** (đi vòng vật cản, không xuyên tòa nhà), **không** phải Manhattan?
- Chỉ **đi tiếp** vào ô `== 0`; tòa nhà `1` và vật cản `2` không bao giờ được đưa vào hàng đợi?
- Kiểm tra biên đủ **hai đầu** `0 <= nr < m and 0 <= nc < n`?
- Hàng đợi là `deque` + `popleft()`, không phải `list.pop(0)`?
- Đánh dấu `seen`/`dist` **ngay khi `append`**, và dùng bảng `seen` **mới** cho mỗi tòa nhà (hoặc mẹo `walk`)?
- Ô mới nhận `d + 1` (không lệch một bước) — Example 2 `[[1, 0]]` ra `1`?
- Có bảng `reach` (hoặc dùng `best` của lần BFS cuối) để loại ô không tới được mọi tòa nhà?
- Chỉ chọn ô `grid[r][c] == 0` làm nơi xây nhà, và trả `-1` (không phải `inf`/`0`) khi không có ô hợp lệ?
- Nếu sửa `grid` làm dấu: đã nghĩ tới việc chép bản sao `[row[:] for row in grid]` chưa?
- Đã test: 3 ví dụ của đề, `wall` (bẫy Manhattan), `[[1,0,1,0,1]]`, `[[1,1,0]]`, `pocket` (bẫy thiếu `reach`), tòa nhà bị bao kín, và một lưới `50 × 50`?